# Corrected probe: pump-field exponent variation

**Restart the kernel and Run All.** The final cell runs **E⁴, E⁸ and E¹²** injection-response cases at **pump CEP = 0**, keeping **del_omega_D = 130 THz** and **gamma_D0 = 250 THz** fixed. Each case uses the same propagation-corrected incident probe. The cells above the final runner define the scientific stages and are skipped during the initial pass.

Only the exponent inside the pump integral changes:
$$
\omega_D(z,t;\Delta)=\omega_{D,0}+\Delta\omega_D
\frac{\sqrt{\int^{t-s(\Delta)} |E_{\rm pump}(z,t')|^p\,dt'}}
{\max_{z'}\sqrt{\int^{t_{\rm end}} |E_{\rm pump}(z',t')|^p\,dt'}},
\qquad p\in\{4,8,12\},\quad\omega_{D,0}=0.
$$
The outer square root is retained. Each exponent has its own global normalization, so del_omega_D remains the largest imposed increase. Spatial differences are retained; normalization is never per cell or per delay. All three powers are even, so $|E|^p=E^p$.

The exponent acts on the recorded pump, so the sweep needs **one shared 3 fs, 1.5 eV, CEP-zero pump propagation**. Each exponent then builds a separate injection cache, runs two probe controls and a complete delay scan, and saves complete-waveform and single-lobe analyses. Existing settings remain **2 nm FDTD cells**, **151 central delay samples**, **30 fs outer echo spacing**, and the current duration rules. Solver units remain ps⁻¹ for gamma and rad/ps for del_omega_D, with no additional 2π conversion.

Outputs are under `lightmatter/output/project2/corrected/exponent_variation/`:

- `pump_3fs_1p5eV_sio2_12p82um/`: the shared CEP-zero pump.
- `E4/`, `E8/`, `E12/`: each contains a scan named `probe_pump_sqrt_E<p>_3fs_1p5eV_gamma_D0_250.0000_ps-1_del_omega_D_130.0000_rad-ps-1_cubic/`, its independent cache, and its experimental comparison.
- `figures/`: copies of pump propagation, waveform diagnostics, integrated-energy and single-lobe plots grouped by exponent.
- `experimental_simulation_comparison/`: combined overlays of all three exponents with experiment, numerical analyses and metadata.

Run All checks every existing case before writing results. It generates the shared pump if absent and reuses only compatible completed pump/scans. Incomplete or incompatible outputs stop the run before overwriting them. Previous CEP and phase-zero results remain in their own directories. Corrected-source provenance, exponent and CEP are validated; experimental comparisons still use the measured transmitted reference and the fixed positive lobe near **33.5 fs**. Delays retain the sampled incident **|E| peak** convention.

The notebook is prepared for execution. No exponent-sweep simulations were started during setup, and previous cell outputs have been cleared.


In [1]:
# Shared corrected exponent-variation settings
# Restart Kernel and Run All; only the final cell executes the pipeline.
MATCHING_RUN_IN_FINAL_CELL = True
MATCHING_GAMMA_D0 = [250.0]
MATCHING_DEL_OMEGA_D = 130.0
MATCHING_PARENT_NAME = "corrected/exponent_variation"
MATCHING_PUMP_NAME = "pump_3fs_1p5eV_sio2_12p82um"
MATCHING_PUMP_PHASE_RAD = 0.0
MATCHING_EXPONENT_CASES = [
    {"label": "E^4", "exponent": 4, "tag": "E4"},
    {"label": "E^8", "exponent": 8, "tag": "E8"},
    {"label": "E^12", "exponent": 12, "tag": "E12"},
]


# Shared CEP-zero pump propagation

Propagate one Gaussian pump with **3 fs intensity-envelope FWHM**, **1.5 eV carrier energy** and **CEP = 0** through the same **12.82 µm fused-silica sample**:
$$
E(t)=E_0\cos[2\pi f_0(t-t_0)]\exp[-(t-t_0)^2/(2\tau^2)],\qquad
\tau=3\,\mathrm{fs}/(2\sqrt{\ln 2}).
$$
Keep $E_0=1$, $t_0=10$ fs and $f_0=(1.5\,\mathrm{eV})/h$ fixed. `PulseParams.phase` is zero radians. The source envelope center and sampled incident peak are recorded in the pump metadata.

The pump uses the fixed three-pole SiO2 background with the additional Drude/Lorentz response disabled. Its propagation is independent of the injection exponent, so all three cases use the same saved pump field. `PUMP_PROBE_EVERY = 9` records every ninth material cell (18 nm on the 2 nm grid) at every time step. Each exponent independently checks interpolation accuracy and builds its response from these recorded fields. Full-domain snapshots are disabled, and pump working arrays are released before the probe scans.


In [2]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    from pathlib import Path
    import json

    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.constants import elementary_charge, h
    from lightmatter.fdtd_revised_sio2 import (
        FDTDParams, PulseParams, MaterialParamsTD, run_simulation,
        SIO2_BACKGROUND_DEL_EPS_DEFAULT, SIO2_BACKGROUND_LAMBDA0_UM_DEFAULT,
    )
    from lightmatter.io import save_simulation_run
    from lightmatter.units import C0, UNIT_T, UNIT_E

    # Resolve output correctly from the workspace, package, or notebook directory.
    pump_search_roots = (Path.cwd(), *Path.cwd().parents)
    pump_package_root = next(
        (candidate for root in pump_search_roots
         for candidate in (root, root / "lightmatter")
         if (candidate / "src/lightmatter/fdtd_revised_sio2.py").is_file()),
        None,
    )
    if pump_package_root is None:
        raise FileNotFoundError("Run this section from inside the lightmatter workspace.")
    pump_output_dir = (
        pump_package_root / "output/project2" / MATCHING_PUMP_PARENT / MATCHING_PUMP_NAME
    )
    # Every exponent uses this same sweep-local pump directory.

    PUMP_INTENSITY_FWHM_FS = 3.0
    PUMP_ENERGY_EV = 1.5
    PUMP_PROBE_EVERY = 9       # 1 = every material cell
    pump_f0 = PUMP_ENERGY_EV * elementary_charge / h * UNIT_T  # cycles/ps
    pump_tau = PUMP_INTENSITY_FWHM_FS * 1e-3 / (2 * np.sqrt(np.log(2)))
    pump_pulse = PulseParams(
        E0=1.0, f0=pump_f0, t0=10e-3, tau=pump_tau, phase=MATCHING_PUMP_PHASE_RAD, type="gaussian",
    )

    # Same geometry as the experimental scan; 2 nm cells resolve the optical pump.
    # dt also resolves the fastest equilibrium UV pole (omega_max*dt < 0.3).
    pump_dz = 20.0                                      # Angstrom
    pump_dt = 0.99 * pump_dz / C0                        # ps
    pump_Nz = int(round(30e4 / pump_dz))
    pump_src_i = int(round(2.5e4 / pump_dz))
    pump_material_cells = int(round(12.82e4 / pump_dz))
    print(f"Number of material cells: {pump_material_cells}")
    pump_metal_i0 = int(round(15e4 / pump_dz)) - pump_material_cells // 2
    pump_metal_i1 = pump_metal_i0 + pump_material_cells
    pump_probe_r_i = pump_metal_i0 - int(round(3e4 / pump_dz))
    pump_probe_t_i = pump_metal_i1 + int(round(3e4 / pump_dz))

    print(f"Last probe id: {np.arange(pump_metal_i0, pump_metal_i1, PUMP_PROBE_EVERY, dtype=np.int64)[-1]}, last SiO2 cell:{pump_metal_i1}")

    # Estimate group delay from the same background used by the solver.
    pump_background_B = np.array(SIO2_BACKGROUND_DEL_EPS_DEFAULT)
    pump_background_lambda_um = np.array(SIO2_BACKGROUND_LAMBDA0_UM_DEFAULT)
    pump_lambda_um = C0 * 1e-4 / pump_f0
    pump_resonance_sq = pump_background_lambda_um**2
    pump_epsilon = 1 + np.sum(
        pump_background_B * pump_lambda_um**2 / (pump_lambda_um**2 - pump_resonance_sq)
    )
    pump_n = np.sqrt(pump_epsilon)
    pump_deps_dlambda = np.sum(
        -2 * pump_background_B * pump_lambda_um * pump_resonance_sq
        / (pump_lambda_um**2 - pump_resonance_sq)**2
    )
    pump_ng = pump_n - pump_lambda_um * pump_deps_dlambda / (2 * pump_n)
    pump_transit_ps = (
        (pump_metal_i0 - pump_src_i) * pump_dz
        + pump_ng * pump_material_cells * pump_dz
        + (pump_probe_t_i - pump_metal_i1) * pump_dz
    ) / C0
    # Record the main transmitted pulse plus 30 fs; later internal echoes need longer runs.
    pump_duration_ps = pump_pulse.t0 + 6 * pump_tau + pump_transit_ps + 30e-3
    pump_Nt = int(np.ceil(pump_duration_ps / pump_dt)) + 1
    pump_time_ps = np.arange(pump_Nt) * pump_dt
    pump_material_z_A = np.arange(pump_metal_i0, pump_metal_i1) * pump_dz
    pump_params = FDTDParams(
        Nz=pump_Nz, Nt=pump_Nt, dz=pump_dz, dt=pump_dt,
        src_i=pump_src_i, probe_t_i=pump_probe_t_i, probe_r_i=pump_probe_r_i,
        metal_i0=pump_metal_i0, metal_i1=pump_metal_i1,
        npml=int(round(1e4 / pump_dz)), snap_every=0,
    )
    pump_material = MaterialParamsTD.from_static(
        Nt=pump_Nt, omega_D=0.0, gamma_D=0.0,
        omega_L=0.0, gamma_L=0.0, del_eps=0.0, eps_inf=1.0,
    )
    pump_source_field = (
        pump_pulse.E0 * np.cos(2 * np.pi * pump_f0 * (pump_time_ps - pump_pulse.t0) + pump_pulse.phase)
        * np.exp(-0.5 * ((pump_time_ps - pump_pulse.t0) / pump_tau)**2)
    )
    pump_nprobes = len(range(pump_metal_i0, pump_metal_i1, PUMP_PROBE_EVERY))
    print(f"Pump CEP: {MATCHING_PUMP_PHASE_LABEL} ({pump_pulse.phase:.12g} rad)")
    print(f"Carrier: {pump_f0:.6f} THz, wavelength: {pump_lambda_um:.6f} um")
    print(f"Intensity FWHM: {PUMP_INTENSITY_FWHM_FS:g} fs; tau: {pump_tau*1e3:.6f} fs")
    print(f"Grid: {pump_Nz} cells, {pump_Nt} steps, dt={pump_dt*1e3:.6f} fs")
    print(f"Material probes: {pump_nprobes}; field storage: {8*pump_Nt*pump_nprobes/2**20:.1f} MiB")
    print(f"Output directory: {pump_output_dir}")

In [3]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Run this cell to simulate and save the pump. The runner checks that its folder is new.
    pump_E_t, pump_E_r, pump_snaps_t, pump_snaps_E, pump_probes = run_simulation(
        pump_params, pump_pulse, pump_material,
        method="sio2_background",
        material_probe_every=PUMP_PROBE_EVERY,
    )
    if not all(np.all(np.isfinite(a)) for a in (pump_E_t, pump_E_r, pump_probes["E"])):
        raise RuntimeError("Pump simulation produced non-finite fields; check the grid/material parameters.")

    # Preserve the project's standard info.json + data.npz format.
    save_simulation_run(
        folder=str(pump_output_dir), params=pump_params, pulse=pump_pulse,
        material=pump_material, E_t=pump_E_t, E_r=pump_E_r,
        snaps_t=pump_snaps_t, snaps_E=pump_snaps_E,
    )
    # Extra time-resolved material probes, with explicit axes and units.
    np.savez_compressed(
        pump_output_dir / "pump_material_probes.npz",
        E=pump_probes["E"], t_ps=pump_probes["t"],
        z_A=pump_probes["z"], z_um=pump_probes["z"]*1e-4,
        cell_indices=pump_probes["indices"], source_E=pump_source_field,
    )
    pump_metadata = {
        "method": "sio2_background",
        "common_envelope_phase_rad": float(pump_pulse.phase),
        "common_envelope_phase_label": MATCHING_PUMP_PHASE_LABEL,
        "envelope_center_fs": float(pump_pulse.t0 * 1e3),
        "incident_abs_peak_source_fs": float(pump_time_ps[np.argmax(np.abs(pump_source_field))] * 1e3),
        "source_phase_convention": "cos(2*pi*f0*(t-t0)+phase); radians",
        "intensity_envelope_fwhm_fs": PUMP_INTENSITY_FWHM_FS,
        "carrier_energy_eV": PUMP_ENERGY_EV,
        "carrier_frequency_THz": pump_f0,
        "carrier_wavelength_um": pump_lambda_um,
        "field_unit": "code units",
        "field_unit_V_per_m": UNIT_E,
        "source_amplitude_note": "Unit soft-source amplitude; no experimental intensity calibration",
        "probe_every_cells": PUMP_PROBE_EVERY,
        "probe_field_axes": ["time", "material_probe"],
        "material_parameter_axes": ["time", "material_cell (if 2D)"],
        "sample_time_convention": "Post-update/source fields labelled by source time n*dt",
        "background_del_eps": pump_background_B.tolist(),
        "background_lambda0_um": pump_background_lambda_um.tolist(),
        "background_gamma_per_ps": [0.0] * len(pump_background_B),
    }
    (pump_output_dir / "pump_metadata.json").write_text(json.dumps(pump_metadata, indent=2))
    print(f"Saved fields, material parameters, probes and metadata to {pump_output_dir}")
    # Reload probes later with np.load(pump_output_dir / "pump_material_probes.npz").

In [4]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Show and save the pump propagation. Thin time samples only for plotting.
    pump_plot_step = max(1, pump_Nt // 2000)
    pump_fig, pump_axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    pump_axes[0].plot(pump_time_ps*1e3, pump_source_field, label="Soft source")
    pump_axes[0].plot(pump_time_ps*1e3, pump_E_t, label="Transmitted probe")
    pump_axes[0].set(xlabel="Simulation time (fs)", ylabel="E (code units)")
    pump_axes[0].legend()
    pump_axes[0].grid(alpha=0.25)
    pump_map = pump_axes[1].pcolormesh(
        pump_probes["z"]*1e-4, pump_probes["t"][::pump_plot_step]*1e3,
        pump_probes["E"][::pump_plot_step], shading="auto", cmap="RdBu_r",
        vmin=-np.max(np.abs(pump_probes["E"])), vmax=np.max(np.abs(pump_probes["E"])),
    )
    pump_axes[1].set(xlabel="Position in simulation domain (um)", ylabel="Simulation time (fs)")
    pump_fig.colorbar(pump_map, ax=pump_axes[1], label="E (code units)")
    pump_fig.suptitle(f"3 fs intensity-FWHM, 1.5 eV pump; CEP = {MATCHING_PUMP_PHASE_LABEL}")
    pump_fig.savefig(pump_output_dir / "pump_propagation.png", dpi=180)
    plt.show()
    # The probe section reloads the saved fields; release the full pump probe array.
    del pump_probes


# Probe pulse simulation: pump-driven carrier injection

The probe source is the **incident field reconstructed from the measured transmitted reference**, read as one trace from `ref_field_corrected.txt` with its companion `time_axis_corrected.txt`. Run `python correct_ref_field.py` in this directory before using this section; the script selects column 0 of `ref_field.txt` by default, removes its baseline once, and inverts the first-pass transfer measured with an unpumped FDTD simulation using the current 2 nm grid. Sending the measured waveform through the sample again from the opposite side alone would not undo its dispersion or transmission amplitude.

The measured record is only about **80 fs**, shorter than the sample's internal round-trip delay. The correction therefore assumes this record describes the first transmitted pass; delayed internal echoes cannot be recovered from that finite record. A constant group transit is removed from the transfer, so the corrected trace retains the original time axis with an arbitrary incident-source origin. The inverse uses a 60–320 THz passband with smooth 40–60 and 320–400 THz edges to avoid amplifying the infrared resonance and out-of-band noise; removed energy and independent forward-FDTD errors are recorded in `ref_field_corrected_metadata.json`. This notebook shifts its start to zero and adds `SOURCE_PREPAD_FS`; pump–probe delays still use incident peaks at the sample front. It retains the existing source peak normalization `E0 = 0.1` and **does not subtract a second baseline**. The metadata and SHA256 hashes are validated before loading, and are saved with each scan. Experimental output comparisons still use the original paired transmitted measurements in `ref_field.txt` and `mod_field.txt`.

The recorded pump field supplies
$$
 Q(z,t)=\int_{t_{\rm record,start}}^t E_{\rm pump}(z,t')^p\,dt',\qquad
 \omega_D(z,t;\Delta)=\underbrace{0}_{\omega_{D,0}}+
 \Delta\omega_D\sqrt{\frac{Q(z,t-s(\Delta))}{\max_{z'}Q(z',t_{\rm record,end})}}.
$$
Here $p$ is the selected pump-field exponent (4, 8 or 12). `del_omega_D` is the **largest increase anywhere in the sample**. The normalization is global within each exponent case, never per position or per delay; spatial fluence differences are retained. The increase in **omega_D** scales with the **square root of the integrated selected even power**. The shared cache stores $R(z,t)=\sqrt{Q(z,t)}$ divided by its global final maximum, so the square root is applied once, after integration and before normalization. `gamma_D` remains constant. This is a prescribed response to the saved pump; the probe does not modify the pump or generate additional carriers.

**Delay convention:** `delay_fs = t_probe_peak_incident - t_pump_peak_incident`. Positive delay means **pump first**. Peaks mean the highest `abs(E)` sample of each incident waveform, before material reshaping. Both are vacuum-propagated from their source to the same sample-front plane. With the probe fixed in solver time,
$$
 s(\Delta)=t_{\rm probe,peak}^{\rm front}-\Delta-t_{\rm pump,peak}^{\rm front},
 \qquad Q_\Delta(z,t)=Q(z,t-s).
$$
An early pump may already have injected carriers at simulation time zero; the injection response is then initialized to its shifted square-root-integral value. It is zero before the saved pump record and stays at its final plateau afterwards (no recombination).

Run All executes the stages below once per exponent through the final runner, loading the shared CEP-zero pump geometry and grid. Completed corrected-source results are reused only when their exponent, zero CEP, configuration and source provenance match. The finite pump record defines the injection model: pump echoes beyond its end are absent. Increase the pump recording duration above if those echoes matter.

In [5]:
# Corrected probe source provenance (read-only).
def load_corrected_probe_provenance(experimental_dir):
    """Validate the correction manifest and identify the exact source artifacts."""
    from pathlib import Path
    import hashlib
    import json

    experimental_dir = Path(experimental_dir)
    field_path = experimental_dir / "ref_field_corrected.txt"
    time_path = experimental_dir / "time_axis_corrected.txt"
    metadata_path = experimental_dir / "ref_field_corrected_metadata.json"
    for path in (field_path, time_path, metadata_path):
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing corrected probe artifact: {path}. Run correct_ref_field.py first."
            )
    metadata = json.loads(metadata_path.read_text())
    trace_index = metadata.get("reference_trace_index")
    if isinstance(trace_index, bool) or not isinstance(trace_index, int) or trace_index < 0:
        raise ValueError("Correction metadata requires a nonnegative reference_trace_index.")
    method = metadata.get("method")
    if not isinstance(method, str) or not method:
        raise ValueError("Correction metadata must identify the inverse-propagation method.")
    field_sha256 = hashlib.sha256(field_path.read_bytes()).hexdigest()
    time_sha256 = hashlib.sha256(time_path.read_bytes()).hexdigest()
    if (metadata.get("corrected_field_sha256") != field_sha256
            or metadata.get("corrected_time_sha256") != time_sha256):
        raise ValueError("Corrected waveform/time hashes disagree with metadata; rerun correct_ref_field.py.")
    provenance = {
        "field_filename": field_path.name,
        "time_filename": time_path.name,
        "metadata_filename": metadata_path.name,
        "corrected_field_sha256": field_sha256,
        "corrected_time_sha256": time_sha256,
        "metadata_sha256": hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
        "reference_trace_index": trace_index,
        "method": method,
    }
    return metadata, provenance


In [6]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Probe configuration and saved pump inputs
    from pathlib import Path
    from dataclasses import asdict
    import json
    import warnings

    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.integrate import cumulative_trapezoid, trapezoid
    from scipy.signal import hilbert
    from tqdm.auto import tqdm
    from lightmatter.fdtd_revised_sio2 import FDTDParams, PulseParams, MaterialParamsTD, run_simulation
    from lightmatter.units import C0, UNIT_T

    package_root = next(
        (candidate for root in (Path.cwd(), *Path.cwd().parents)
         for candidate in (root, root / "lightmatter")
         if (candidate / "src/lightmatter/fdtd_revised_sio2.py").is_file()), None,
    )
    if package_root is None:
        raise FileNotFoundError("Run this notebook from inside the lightmatter workspace.")
    experimental_dir = package_root / "notebooks/project_2"
    PUMP_RUN_DIR = package_root / "output/project2" / MATCHING_PUMP_PARENT / MATCHING_PUMP_NAME
    if isinstance(PUMP_FIELD_EXPONENT, bool) or PUMP_FIELD_EXPONENT not in (4, 8, 12):
        raise ValueError("Choose pump-field exponent 4, 8, or 12.")
    SPATIAL_INTERPOLATION = "cubic"     # "cubic", "linear", "retarded", or "none"
    PROBE_DZ_A = None                  # None uses this batch's pump grid (default 2 nm)
    INJECTION_CACHE_DT_FS = 0.02       # integrate at original pump dt; then cache sqrt(cumulative E^p)
    SPATIAL_BLOCK_CELLS = 32          # bounds temporary field/interpolation arrays

    SOURCE_PREPAD_FS = 20.0
    POST_RECORD_MARGIN_FS = 30.0       # minimum; also allow echoes and Drude decay
    MIN_INTERNAL_ROUND_TRIPS = 0.5     # this is also a safety feature, but I don't care too much about internal reflections
    MIN_DRUDE_DAMPING_TIMES = 1.0      # this is probably not theoretically enough, but this scales very badly

    ENERGY_TAIL_FRACTION = 1e-3        # central 99.8% probe energy / local pump sqrt(integral E^p) response
    SCAN_MARGIN_FS = 20.0
    N_DELAYS = 151                    # dense samples across the main-pulse overlap
    ECHO_DELAY_STEP_FS = 30.0          # coarser outer scan covering weaker internal echoes
    PROBE_OCCUPANCY_TAIL_FRACTION = 1e-3
    PROBE_TIMING_EVERY_CELLS = 32     # sparse probes in the unpumped control, including both ends
    REPRESENTATIVE_DELAY_FS = 0.0
    CARRIER_FREQUENCY_THz = 166.0
    TAIL_CHECK_FS = 10.0
    TAIL_ENERGY_TOLERANCE = 1e-4
    INTERPOLATION_RESPONSE_TOLERANCE = 0.01

    ########---------------- TIME-VARYING DRUDE PARAMETERS ----------------########
    # Values follow the existing solver convention: ps^-1 for gamma, rad/ps for
    # del_omega_D (numerically THz); no additional 2*pi conversion.
    omega_D0 = 0.0
    gamma_D0 = MATCHING_GAMMA_D0[0]  # held fixed throughout the exponent sweep
    del_omega_D = MATCHING_DEL_OMEGA_D
    ########---------------- ----------------------------- ----------------########

    # Validate source artifacts before loading potentially large saved pump arrays.
    corrected_source_metadata, probe_source_provenance = load_corrected_probe_provenance(experimental_dir)

    with (PUMP_RUN_DIR / "info.json").open() as stream:
        saved_pump_info = json.load(stream)
    with (PUMP_RUN_DIR / "pump_metadata.json").open() as stream:
        saved_pump_metadata = json.load(stream)
    if (not np.isclose(saved_pump_info["pulse_params"]["phase"], MATCHING_PUMP_PHASE_RAD, rtol=0, atol=1e-12)
            or not np.isclose(saved_pump_metadata["common_envelope_phase_rad"], MATCHING_PUMP_PHASE_RAD, rtol=0, atol=1e-12)):
        raise ValueError("Saved pump CEP differs from this case.")
    saved_pump_params = FDTDParams(**saved_pump_info["fdtd_params"])
    with np.load(PUMP_RUN_DIR / "pump_material_probes.npz") as archive:
        pump_record_t = archive["t_ps"]
        pump_record_z = archive["z_A"]
        pump_record_E = archive["E"]
        pump_record_source = archive["source_E"]
        pump_record_indices = archive["cell_indices"]
    if (pump_record_t.ndim != 1 or pump_record_z.ndim != 1
            or pump_record_E.shape != (pump_record_t.size, pump_record_z.size)
            or pump_record_source.shape != pump_record_t.shape
            or pump_record_t.size < 3 or pump_record_z.size < 1):
        raise ValueError("Pump arrays must be t:(Nt,), z:(Nprobes,), E:(Nt,Nprobes), source:(Nt,).")
    if not all(np.all(np.isfinite(a)) for a in (pump_record_t, pump_record_z, pump_record_E, pump_record_source)):
        raise ValueError("Saved pump arrays contain non-finite values.")
    if np.any(np.diff(pump_record_t) <= 0) or np.any(np.diff(pump_record_z) <= 0):
        raise ValueError("Pump time and probe positions must be strictly increasing.")
    if (pump_record_t.size != saved_pump_params.Nt
            or not np.allclose(pump_record_t, np.arange(saved_pump_params.Nt)*saved_pump_params.dt)
            or not np.array_equal(pump_record_z, pump_record_indices*saved_pump_params.dz)
            or np.any(pump_record_indices < saved_pump_params.metal_i0)
            or np.any(pump_record_indices >= saved_pump_params.metal_i1)):
        raise ValueError("Pump probe axes disagree with the saved geometry/timestep.")
    if not np.any(pump_record_source):
        raise ValueError("The saved incident pump waveform is zero.")
    if SPATIAL_INTERPOLATION not in ("none", "linear", "cubic", "retarded"):
        raise ValueError("Unknown spatial interpolation mode.")
    if not (0 < ENERGY_TAIL_FRACTION < 0.5) or INJECTION_CACHE_DT_FS <= 0:
        raise ValueError("Require 0 < ENERGY_TAIL_FRACTION < 0.5 and a positive cache timestep.")
    if not isinstance(N_DELAYS, int) or N_DELAYS < 3 or del_omega_D <= 0 or gamma_D0 < 0:
        raise ValueError("Require at least three delays, positive del_omega_D, and nonnegative damping.")
    if ECHO_DELAY_STEP_FS <= 0 or not (0 < PROBE_OCCUPANCY_TAIL_FRACTION < 0.5):
        raise ValueError("Require positive echo delay spacing and a valid probe occupation tail fraction.")
    if not isinstance(PROBE_TIMING_EVERY_CELLS, int) or PROBE_TIMING_EVERY_CELLS < 1:
        raise ValueError("PROBE_TIMING_EVERY_CELLS must be a positive integer.")
    if MIN_INTERNAL_ROUND_TRIPS < 0 or MIN_DRUDE_DAMPING_TIMES < 0:
        raise ValueError("Echo and damping-time allowances must be nonnegative.")
    if not isinstance(SPATIAL_BLOCK_CELLS, int) or SPATIAL_BLOCK_CELLS < 1:
        raise ValueError("SPATIAL_BLOCK_CELLS must be a positive integer.")
    if SOURCE_PREPAD_FS < 0 or POST_RECORD_MARGIN_FS <= 0 or SCAN_MARGIN_FS < 0:
        raise ValueError("Padding/margins must be nonnegative, with a positive recording margin.")

    # One reconstructed incident trace; the correction script already removed the baseline.
    time_axis_fs = np.asarray(np.loadtxt(experimental_dir / "time_axis_corrected.txt", delimiter=","), dtype=float)
    reference_field_corrected = np.asarray(np.loadtxt(
        experimental_dir / "ref_field_corrected.txt", delimiter=",",
    ), dtype=float)
    if (time_axis_fs.ndim != 1 or reference_field_corrected.shape != time_axis_fs.shape
            or time_axis_fs.size < 3 or np.any(np.diff(time_axis_fs) <= 0)
            or not np.all(np.isfinite(time_axis_fs)) or not np.all(np.isfinite(reference_field_corrected))):
        raise ValueError("Corrected time and field must be finite, equally sized 1D arrays with increasing times.")
    waveform_time_rel_fs = time_axis_fs - time_axis_fs[0]
    waveform_time_rel_ps = waveform_time_rel_fs * 1e-3
    experimental_dt_fs = float(np.median(np.diff(time_axis_fs)))
    if not np.allclose(np.diff(time_axis_fs), experimental_dt_fs, rtol=1e-6, atol=1e-12):
        raise ValueError("Corrected source sampling must be uniform for the bandwidth estimate.")
    E0 = 0.1
    reference_peak = float(np.max(np.abs(reference_field_corrected)))
    if reference_peak == 0:
        raise ValueError("The corrected incident probe is zero.")
    reference_field = reference_field_corrected * (E0 / reference_peak)
    probe_incident_peak_rel_ps = waveform_time_rel_ps[np.argmax(np.abs(reference_field))]
    pump_incident_peak_source_ps = pump_record_t[np.argmax(np.abs(pump_record_source))]
    if max(abs(reference_field[0]), abs(reference_field[-1])) > 0.01 * E0:
        warnings.warn("Probe endpoints exceed 1% of peak; zero-padding may introduce a discontinuity.")
    print(f"Corrected incident record: {waveform_time_rel_fs[-1]:.3f} fs; dt={experimental_dt_fs:.6f} fs")
    print(f"Incident probe |E| peak: {probe_incident_peak_rel_ps*1e3:.6f} fs from record start")
    print(f"Loaded {pump_record_E.shape[1]} pump probes from {PUMP_RUN_DIR}")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(waveform_time_rel_fs, reference_field, label="Corrected incident probe (peak = 0.1)")
    ax.set(xlabel="Source time from corrected record start (fs)", ylabel="Source field (code units)",
           title=f"Corrected incident reference (original measured trace {corrected_source_metadata['reference_trace_index']})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## Interpolate the field first, then take the square root of its integrated selected even power

The default is a **local four-point cubic polynomial in position**, evaluated on the electric fields at each recorded time. The spatial weights are reused for all times. At a recorded position the original trace is recovered exactly. This retains both forward and reflected components and the dispersion already present in the recorded data, provided the probes resolve the spatial oscillations. It avoids interpolating the nonlinear quantity $E^p$ between probes. See the [SciPy discussion of polynomial interpolation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.BarycentricInterpolator.html) for the underlying interpolation principle; only four local points are used here.

Optional `retarded` interpolation uses the two neighbouring analytic field traces, shifts their envelopes at the pump group velocity, corrects carrier phase using the phase index, and takes a weighted average. For displacement $d$ from a neighbour, its contribution is
$\Re\{E_a(t-n_gd/c)\exp[i\omega_0(n_g-n)d/c]\}$.
It is a forward-wave, first-order dispersion approximation; reflected waves and broad bandwidth can reduce its accuracy. The cubic method needs no propagation-velocity approximation. Both methods are checked below against withheld recorded probes, including the resulting $\sqrt{\int E^pdt}$ error. With very coarse probes, neither can recover missing spatial information: rerun the pump with finer probe spacing. `none` requires a recorded probe at every requested material position.

The last regular probe may precede the last material cell. A one-sided cubic/linear continuation is allowed only within one probe spacing and only inside the saved material. Edge reconstruction is less constrained than interpolation; record the last cell explicitly for stringent interface studies.

In [7]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Numerical helpers: spatial fields, cumulative integrals, and time shifts

    def silica_indices(frequency_THz):
        """Phase/group indices of the saved lossless equilibrium background."""
        f = np.asarray(frequency_THz, dtype=float)
        if np.any(f <= 0):
            raise ValueError("Frequency must be positive.")
        B = np.asarray(saved_pump_metadata["background_del_eps"])
        resonances = np.asarray(saved_pump_metadata["background_lambda0_um"])**2
        wavelength = C0 * 1e-4 / f
        denom = wavelength[..., None]**2 - resonances
        eps = 1 + np.sum(B * wavelength[..., None]**2 / denom, axis=-1)
        deriv = np.sum(-2*B*wavelength[..., None]*resonances/denom**2, axis=-1)
        if not np.all(np.isfinite(eps)) or np.any(eps <= 0):
            raise ValueError("Selected frequencies are outside the transparent silica band.")
        n = np.sqrt(eps)
        ng = n - wavelength * deriv / (2*n)
        return n, ng


    def pump_injection_response(field, time, axis=0):
        """Unnormalized Drude increase: sqrt(integral |E|^p dt), initially zero."""
        if (isinstance(PUMP_FIELD_EXPONENT, bool)
                or not isinstance(PUMP_FIELD_EXPONENT, (int, np.integer))
                or PUMP_FIELD_EXPONENT not in (4, 8, 12)):
            raise ValueError("Choose pump-field exponent 4, 8, or 12.")
        return np.sqrt(cumulative_trapezoid(
            np.abs(np.asarray(field, dtype=float))**PUMP_FIELD_EXPONENT, time, axis=axis, initial=0,
        ))


    def integral_quantiles(time, cumulative, quantiles):
        """Times at given fractions of a nondecreasing integral or injection response."""
        if cumulative[-1] <= 0:
            return np.full(len(quantiles), np.nan)
        return np.interp(np.asarray(quantiles)*cumulative[-1], cumulative, time)


    def interpolate_pump_fields(time, positions, fields, targets, mode="cubic",
                                phase_index=None, group_index=None, carrier_THz=None):
        """Return E(time, target); call on spatial blocks to bound working memory."""
        time, positions, targets = map(lambda x: np.asarray(x, dtype=float), (time, positions, targets))
        fields = np.asarray(fields)
        if (time.ndim != 1 or positions.ndim != 1 or targets.ndim != 1
                or fields.shape != (time.size, positions.size)
                or time.size < 2 or positions.size == 0
                or np.any(np.diff(time) <= 0) or np.any(np.diff(positions) <= 0)
                or not all(np.all(np.isfinite(a)) for a in (time, positions, targets))):
            raise ValueError("Invalid field interpolation dimensions/coordinates.")
        if mode == "none":
            idx = np.searchsorted(positions, targets)
            if np.any(idx >= positions.size) or not np.allclose(positions[np.minimum(idx, positions.size-1)], targets, rtol=0, atol=1e-8):
                raise ValueError("Interpolation is disabled: pump probes must cover every requested cell.")
            return fields[:, idx].copy()
        if mode not in ("linear", "cubic", "retarded"):
            raise ValueError("Unknown interpolation mode.")
        count = 4 if mode == "cubic" else 2
        if positions.size < count:
            raise ValueError(f"{mode} interpolation needs at least {count} recorded probes.")
        if np.any(targets < positions[0] - (positions[1]-positions[0])) or np.any(targets > positions[-1] + (positions[-1]-positions[-2])):
            raise ValueError("Refusing extrapolation by more than one probe spacing.")
        starts = np.clip(np.searchsorted(positions, targets) - count//2, 0, positions.size-count)
        ids = starts[:, None] + np.arange(count)[None, :]
        x = positions[ids]
        weights = np.ones_like(x)
        for j in range(count):
            for k in range(count):
                if k != j:
                    weights[:, j] *= (targets-x[:, k])/(x[:, j]-x[:, k])
        out = np.zeros((time.size, targets.size))
        if mode != "retarded":
            for j in range(count):
                out += fields[:, ids[:, j]] * weights[:, j]
            return out
        if (phase_index is None or group_index is None or carrier_THz is None
                or min(phase_index, group_index, carrier_THz) <= 0):
            raise ValueError("Retarded interpolation requires positive phase/group indices and carrier frequency.")
        # Pad both ends before Hilbert transformation to reduce periodic edge artifacts.
        selected, inverse = np.unique(ids, return_inverse=True)
        analytic = hilbert(np.pad(fields[:, selected], ((time.size, time.size), (0, 0))), axis=0)[time.size:2*time.size]
        lookup = inverse.reshape(ids.shape)
        for col, target in enumerate(targets):
            for j in range(count):
                displacement = target - x[col, j]
                shifted_time = time - group_index*displacement/C0
                a = analytic[:, lookup[col, j]]
                shifted = (np.interp(shifted_time, time, a.real, left=0, right=0)
                           + 1j*np.interp(shifted_time, time, a.imag, left=0, right=0))
                phase = 2*np.pi*carrier_THz*(group_index-phase_index)*displacement/C0
                out[:, col] += weights[col, j] * np.real(shifted*np.exp(1j*phase))
        return out


    def resample_cumulative(time, profile, query, out=None, scale=1.0, baseline=0.0):
        """Linear time interpolation with zero/terminal plateaus, in row blocks."""
        time, query = np.asarray(time), np.asarray(query)
        if (time.ndim != 1 or query.ndim != 1 or profile.ndim != 2
                or profile.shape[0] != time.size or time.size < 2
                or np.any(np.diff(time) <= 0) or not np.all(np.isfinite(query))):
            raise ValueError("Invalid cumulative-profile dimensions or times.")
        if out is None:
            out = np.empty((query.size, profile.shape[1]), dtype=np.float64)
        if out.shape != (query.size, profile.shape[1]) or out.dtype != np.float64:
            raise ValueError("Output must be float64 with shape (Nquery, Nmaterial).")
        for lo in range(0, query.size, 256):
            hi = min(lo+256, query.size)
            index = np.clip(np.searchsorted(time, query[lo:hi], side="right")-1, 0, time.size-2)
            fraction = np.clip((query[lo:hi]-time[index])/(time[index+1]-time[index]), 0, 1)
            out[lo:hi] = baseline + scale*((1-fraction[:, None])*profile[index] + fraction[:, None]*profile[index+1])
        return out

In [8]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Geometry and interpolation cross-validation (no FDTD runs here)
    pump_phase_index, pump_group_index = map(float, silica_indices(saved_pump_metadata["carrier_frequency_THz"]))
    source_prepad = SOURCE_PREPAD_FS*1e-3
    dz = saved_pump_params.dz if PROBE_DZ_A is None else float(PROBE_DZ_A)
    if not np.isfinite(dz) or dz <= 0:
        raise ValueError("PROBE_DZ_A must be positive and finite.")
    dt = min(0.99*dz/C0, saved_pump_params.dt)
    L = saved_pump_params.Nz * saved_pump_params.dz
    Nz = int(round(L/dz))

    def aligned_index(position_A):
        index = int(round(position_A/dz))
        if not np.isclose(index*dz, position_A, atol=1e-8, rtol=0):
            raise ValueError("The chosen probe grid must align with the saved sample/source/probes.")
        return index

    src_i = aligned_index(saved_pump_params.src_i*saved_pump_params.dz)
    probe_r_i = aligned_index(saved_pump_params.probe_r_i*saved_pump_params.dz)
    probe_t_i = aligned_index(saved_pump_params.probe_t_i*saved_pump_params.dz)
    metal_i0 = aligned_index(saved_pump_params.metal_i0*saved_pump_params.dz)
    metal_i1 = aligned_index(saved_pump_params.metal_i1*saved_pump_params.dz)
    material_z_A = np.arange(metal_i0, metal_i1)*dz
    metal_cells = material_z_A.size
    src_pos, probe_r_pos, probe_t_pos = np.array([src_i, probe_r_i, probe_t_i])*dz
    sample_front_pos, sample_back_pos = np.array([metal_i0, metal_i1])*dz
    thickness = sample_back_pos-sample_front_pos
    source_to_front_time = (sample_front_pos-src_pos)/C0
    back_to_transmission_probe_time = (probe_t_pos-sample_back_pos)/C0
    pump_incident_peak_front_ps = pump_incident_peak_source_ps + (
        sample_front_pos-saved_pump_params.src_i*saved_pump_params.dz)/C0
    probe_incident_peak_front_ps = source_prepad + probe_incident_peak_rel_ps + source_to_front_time


    def pump_time_shift_ps(delay_fs):
        return probe_incident_peak_front_ps - float(delay_fs)*1e-3 - pump_incident_peak_front_ps

    interpolation_errors = {}
    # Leave out sparse interior nodes simultaneously. Retain their four neighbours.
    withheld = np.unique(np.linspace(2, pump_record_z.size-3, min(24, max(0, (pump_record_z.size-4)//5))).astype(int))
    if withheld.size and pump_record_z.size >= 8:
        keep = np.ones(pump_record_z.size, dtype=bool)
        keep[withheld] = False
        truth = pump_record_E[:, withheld]
        truth_response = pump_injection_response(truth, pump_record_t)[-1]
        for mode in ("linear", "cubic", "retarded"):
            estimate = interpolate_pump_fields(
                pump_record_t, pump_record_z[keep], pump_record_E[:, keep], pump_record_z[withheld], mode,
                pump_phase_index, pump_group_index, saved_pump_metadata["carrier_frequency_THz"],
            )
            relative_field = np.linalg.norm(estimate-truth, axis=0)/np.maximum(np.linalg.norm(truth, axis=0), np.finfo(float).tiny)
            relative_response = abs(pump_injection_response(estimate, pump_record_t)[-1]/np.maximum(truth_response, np.finfo(float).tiny)-1)
            interpolation_errors[mode] = {"max_field_relative_L2": float(relative_field.max()), "max_response_relative_error": float(relative_response.max())}
            print(f"{mode:8s}: held-out field L2 error <= {relative_field.max():.3%}; sqrt(integral E^{PUMP_FIELD_EXPONENT}) response error <= {relative_response.max():.3%}")
        if SPATIAL_INTERPOLATION != "none" and interpolation_errors[SPATIAL_INTERPOLATION]["max_response_relative_error"] > INTERPOLATION_RESPONSE_TOLERANCE:
            raise ValueError(f"Chosen interpolation failed the sqrt(integral E^{PUMP_FIELD_EXPONENT}) response tolerance. Use finer pump probes or explicitly revise the tolerance after checking convergence.")
    else:
        warnings.warn("Too few pump probes for the held-out accuracy check; verify spatial convergence separately.")
    # A regular probe lattice can leave a short gap before the last material cell.
    # Predict the last recorded trace from its preceding neighbours to test this
    # one-sided continuation separately from interpolation in the interior.
    if material_z_A[-1] > pump_record_z[-1] and pump_record_z.size >= 5 and SPATIAL_INTERPOLATION != "none":
        edge_estimate = interpolate_pump_fields(
            pump_record_t, pump_record_z[-5:-1], pump_record_E[:, -5:-1], pump_record_z[-1:],
            SPATIAL_INTERPOLATION, pump_phase_index, pump_group_index,
            saved_pump_metadata["carrier_frequency_THz"],
        )[:, 0]
        edge_response_error = float(abs(pump_injection_response(edge_estimate, pump_record_t)[-1]/pump_injection_response(pump_record_E[:, -1], pump_record_t)[-1]-1))
        interpolation_errors.setdefault(SPATIAL_INTERPOLATION, {})["back_edge_response_relative_error"] = edge_response_error
        print(f"One-sided back-edge sqrt(integral E^{PUMP_FIELD_EXPONENT}) response check: {edge_response_error:.3%} relative error")
        if edge_response_error > INTERPOLATION_RESPONSE_TOLERANCE:
            raise ValueError("Back-edge reconstruction failed tolerance; record the final material cell or use finer pump probes.")


## Cache the pump-driven injection and choose the delay window

$E^p$ is integrated on the **original pump timestep**, then its square root is taken before the response curves are resampled onto the cache grid. A separate global normalization for each exponent sets that case's largest final response to one; `del_omega_D` therefore remains the maximum increase. The cache and one reusable per-delay Drude array are disk-backed, and field reconstruction is done in spatial blocks. Lower `INJECTION_CACHE_DT_FS` to the pump timestep to eliminate response-curve downsampling. The cell checks the interpolation error introduced by this downsampling, including the last few material cells.

For each material position, the pump injection interval is obtained from the local square-root-response fractions (a fraction $q$ of the final response corresponds to a fraction $q^2$ of the local $\int E^pdt$). An initial main-pulse probe interval is its incident energy-quantile interval plus an unpumped group-delay estimate. Group indices over the significant measured probe spectrum bound this estimate. **The final scan bounds are then refined using recorded interior fields from the unpumped reference**, so that internal reflections within the simulation window are included. Their local energy-quantile intervals use `PROBE_OCCUPANCY_TAIL_FRACTION=1e-3`; the timing probes include both sample ends. If these intervals, relative to the respective incident peak, are $[p_-(z),p_+(z)]$ and $[r_-(z),r_+(z)]$, all local overlap lies in
$$
 \min_z[p_-(z)-r_+(z)]\ \leq\ \Delta\ \leq\ \max_z[p_+(z)-r_-(z)].
$$
A margin is added on both sides. `N_DELAYS` dense points cover the main-pulse overlap, with `ECHO_DELAY_STEP_FS` spacing in the outer echo-sensitive intervals. Zero delay and both final bounds are included exactly. The negative end approaches the unpumped result; the positive end approaches the fully injected spatial profile. These limits refer to the chosen tail tolerance, not mathematically compact pulses.

The probe source remains fixed as delay changes, so early pump propagation need not be rerun. The common simulation duration covers the **full imported probe record**, the slowest retained unpumped group transit, and a post-recording margin. By default this margin is the largest of 30 fs, **0.5 internal round-trip times**, and **one Drude damping time** ($1/\gamma_D$, omitted for zero damping). The echo allowance matters because a quiet short tail window can occur between reflections. Strong time-dependent Drude response can create longer tails: every saved run reports its last-window transmitted-energy fraction, and the scan warns if a longer duration is needed. Later internal echoes also require a longer record. This is a practical initial duration, to be checked by convergence, not a universal optimum.

The actual duration and delay bounds are computed from the saved pump and the current settings below, then recorded in each scan configuration.


In [9]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Build the shared normalized pump-response cache once.
    base_dir = package_root / "output/project2" / MATCHING_CASE_PARENT / (
        f"probe_pump_sqrt_E{PUMP_FIELD_EXPONENT}_3fs_1p5eV_gamma_D0_{gamma_D0:.4f}_ps-1_del_omega_D_{del_omega_D:.4f}_rad-ps-1_{SPATIAL_INTERPOLATION}"
    )
    base_dir.mkdir(parents=True, exist_ok=True)
    # Rebuilding shared material data invalidates any previous completed scan.
    old_manifest = base_dir / "scan_manifest.json"
    if old_manifest.exists():
        old_manifest.write_text(json.dumps({"complete": False, "runs": [], "completed_indices": [],
                                           "reason": "Shared injection cache rebuilt; rerun the scan."}, indent=2))
    shared_dir = base_dir / "shared_injection"
    shared_dir.mkdir(exist_ok=True)
    cache_stride = max(1, int(round(INJECTION_CACHE_DT_FS/(saved_pump_params.dt*1e3))))
    cache_rows = np.unique(np.r_[np.arange(0, pump_record_t.size, cache_stride), pump_record_t.size-1])
    injection_time_ps = pump_record_t[cache_rows]
    cache_path = shared_dir / f"normalized_sqrt_cumulative_E{PUMP_FIELD_EXPONENT}.npy"
    injection_fraction = np.lib.format.open_memmap(
        cache_path, mode="w+", dtype=np.float64, shape=(injection_time_ps.size, metal_cells),
    )
    final_response = np.zeros(metal_cells)
    pump_injection_lo_ps = np.zeros(metal_cells)
    pump_injection_hi_ps = np.zeros(metal_cells)
    cache_time_error = 0.0
    record_tail_fraction = 0.0
    for lo in tqdm(range(0, metal_cells, SPATIAL_BLOCK_CELLS), desc=f"Computing sqrt(integral pump E^{PUMP_FIELD_EXPONENT})"):
        hi = min(lo+SPATIAL_BLOCK_CELLS, metal_cells)
        local_E = interpolate_pump_fields(
            pump_record_t, pump_record_z, pump_record_E, material_z_A[lo:hi], SPATIAL_INTERPOLATION,
            pump_phase_index, pump_group_index, saved_pump_metadata["carrier_frequency_THz"],
        )
        local_response = pump_injection_response(local_E, pump_record_t)
        injection_fraction[:, lo:hi] = local_response[cache_rows]
        final_response[lo:hi] = local_response[-1]
        for j in range(hi-lo):
            pump_injection_lo_ps[lo+j], pump_injection_hi_ps[lo+j] = integral_quantiles(
                pump_record_t, local_response[:, j], (ENERGY_TAIL_FRACTION, 1-ENERGY_TAIL_FRACTION),
            )
        reconstructed = resample_cumulative(injection_time_ps, local_response[cache_rows], pump_record_t)
        cache_time_error = max(cache_time_error, float(np.max(abs(reconstructed-local_response))))
        tail_row = np.searchsorted(pump_record_t, pump_record_t[-1]-5e-3)
        record_tail_fraction = max(record_tail_fraction, float(np.max(
            (local_response[-1]-local_response[tail_row])/np.maximum(local_response[-1], np.finfo(float).tiny))))
        del local_E, local_response, reconstructed
    response_normalization = float(final_response.max())
    if not np.isfinite(response_normalization) or response_normalization <= 0:
        raise ValueError("Pump sqrt(integral E^p) response is zero or non-finite.")
    for lo in range(0, injection_time_ps.size, 256):
        injection_fraction[lo:lo+256] /= response_normalization
    injection_fraction.flush()
    cache_time_error /= response_normalization
    if cache_time_error > 1e-3:
        raise ValueError("Injection-response time interpolation error exceeds 0.1% of global maximum; reduce INJECTION_CACHE_DT_FS.")
    if record_tail_fraction > ENERGY_TAIL_FRACTION:
        warnings.warn("Significant injection remains in the last 5 fs of the pump record; increase the pump simulation duration.")
    np.savez_compressed(
        shared_dir / "axes_and_timing.npz", t_ps=injection_time_ps, z_A=material_z_A,
        final_fraction=final_response/response_normalization,
        pump_injection_lo_ps=pump_injection_lo_ps, pump_injection_hi_ps=pump_injection_hi_ps,
        pump_source_t_ps=pump_record_t, pump_source_E=pump_record_source,
        probe_source_t_rel_ps=waveform_time_rel_ps, probe_source_E=reference_field,
    )
    print(f"Global sqrt(integral E^{PUMP_FIELD_EXPONENT}) normalization: {response_normalization:.8g} (field code units)^{PUMP_FIELD_EXPONENT//2} sqrt(ps)")
    print(f"Injection-response cache interpolation error <= {cache_time_error:.3%} of global maximum")
    print(f"Largest final-5-fs pump injection fraction: {record_tail_fraction:.3%}")
    print(f"Shared profile cache: {injection_fraction.nbytes/2**30:.2f} GiB")

    # Probe bandwidth and conservative group-delay estimates.
    probe_cumulative_energy = cumulative_trapezoid(reference_field**2, waveform_time_rel_ps, initial=0)
    t_wave_lo_rel, t_wave_hi_rel = integral_quantiles(
        waveform_time_rel_ps, probe_cumulative_energy, (ENERGY_TAIL_FRACTION, 1-ENERGY_TAIL_FRACTION),
    )
    probe_frequency = np.fft.rfftfreq(reference_field.size, experimental_dt_fs*1e-3)
    probe_spectral_energy = abs(np.fft.rfft(reference_field))**2
    probe_spectral_energy[0] = 0.0
    probe_cdf = np.cumsum(probe_spectral_energy)/probe_spectral_energy.sum()
    frequency_low, frequency_high = np.interp(
        [ENERGY_TAIL_FRACTION, 1-ENERGY_TAIL_FRACTION], probe_cdf, probe_frequency,
    )
    band = (probe_frequency > 0) & (probe_frequency >= frequency_low) & (probe_frequency <= frequency_high)
    if not np.any(band):
        raise ValueError("No frequencies in the selected probe bandwidth.")
    n_phase, n_group = map(float, silica_indices(CARRIER_FREQUENCY_THz))
    _, band_group_indices = silica_indices(probe_frequency[band])
    ng_min = min(float(band_group_indices.min()), n_group)
    ng_max = max(float(band_group_indices.max()), n_group)
    if ng_min <= 0 or not np.isfinite(ng_max):
        raise ValueError("Invalid group-delay estimate for the probe bandwidth.")
    local_depth = material_z_A-sample_front_pos
    probe_lo_from_peak = t_wave_lo_rel-probe_incident_peak_rel_ps + ng_min*local_depth/C0
    probe_hi_from_peak = t_wave_hi_rel-probe_incident_peak_rel_ps + ng_max*local_depth/C0
    pump_lo_from_peak = pump_injection_lo_ps-pump_incident_peak_front_ps
    pump_hi_from_peak = pump_injection_hi_ps-pump_incident_peak_front_ps
    active_cells = np.isfinite(pump_lo_from_peak) & np.isfinite(pump_hi_from_peak)
    delay_min_fs = float(np.min((pump_lo_from_peak-probe_hi_from_peak)[active_cells])*1e3-SCAN_MARGIN_FS)
    delay_max_fs = float(np.max((pump_hi_from_peak-probe_lo_from_peak)[active_cells])*1e3+SCAN_MARGIN_FS)
    main_delay_min_fs, main_delay_max_fs = delay_min_fs, delay_max_fs
    # The reference simulation below supplies final bounds including internal echoes.
    sample_group_transit_time = n_group*thickness/C0
    source_to_transmission_time = source_to_front_time + sample_group_transit_time + back_to_transmission_probe_time
    source_waveform_end_time = source_prepad + waveform_time_rel_ps[-1]
    round_trip_ps = 2*ng_max*thickness/C0
    record_margin_ps = max(
        POST_RECORD_MARGIN_FS*1e-3,
        MIN_INTERNAL_ROUND_TRIPS*round_trip_ps,
        MIN_DRUDE_DAMPING_TIMES/gamma_D0 if gamma_D0 > 0 else 0.0,
    )
    T_required = source_waveform_end_time + source_to_front_time + ng_max*thickness/C0 + back_to_transmission_probe_time + record_margin_ps
    Nt = int(np.ceil(T_required/dt))+1
    t_ax = np.arange(Nt)*dt
    waveform_padded = np.interp(t_ax, source_prepad+waveform_time_rel_ps, reference_field, left=0, right=0)
    pulse = PulseParams(E0=E0, f0=CARRIER_FREQUENCY_THz, t0=source_prepad+probe_incident_peak_rel_ps,
                        tau=0.5*waveform_time_rel_ps[-1], type="explicit")
    simulation_params_withfilm = FDTDParams(
        Nz=Nz, Nt=Nt, dz=dz, dt=dt, src_i=src_i, probe_t_i=probe_t_i, probe_r_i=probe_r_i,
        metal_i0=metal_i0, metal_i1=metal_i1,
        npml=aligned_index(saved_pump_params.npml*saved_pump_params.dz),
        R_target=saved_pump_params.R_target, m=saved_pump_params.m, snap_every=0,
    )
    gamma_D_constant = np.full(Nt, gamma_D0)
    print(f"Probe bandwidth estimate: {frequency_low:.2f}--{frequency_high:.2f} THz; n_g={ng_min:.6f}--{ng_max:.6f}")
    print(f"Incident front peaks at zero shift: probe={probe_incident_peak_front_ps*1e3:.3f} fs, saved pump={pump_incident_peak_front_ps*1e3:.3f} fs")
    print(f"Initial main-pulse overlap window: {main_delay_min_fs:.3f} to {main_delay_max_fs:.3f} fs")
    print(f"Recording margin: {record_margin_ps*1e3:.3f} fs (echo and damping allowances included)")
    print(f"Run duration: {t_ax[-1]*1e3:.3f} fs, dt={dt*1e3:.6f} fs, Nt={Nt}")
    print(f"Reusable omega_D buffer: {8*Nt*metal_cells/2**30:.2f} GiB; not stored separately for each delay")

In [10]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Incident timing diagnostic and representative injection curves.

    diagnostic_cells = np.unique(np.round(np.linspace(0, metal_cells-1, 10)).astype(int))
    print(f"last cell id : {metal_cells-1}")
    diagnostic_z = material_z_A[diagnostic_cells]
    representative_omega_D = resample_cumulative(
        injection_time_ps, injection_fraction[:, diagnostic_cells],
        t_ax-pump_time_shift_ps(REPRESENTATIVE_DELAY_FS), scale=del_omega_D, baseline=omega_D0,
    )
    probe_front_field = np.interp(t_ax-source_to_front_time, source_prepad+waveform_time_rel_ps, reference_field, left=0, right=0)
    pump_front_field = np.interp(
        t_ax-pump_time_shift_ps(REPRESENTATIVE_DELAY_FS)-(sample_front_pos-saved_pump_params.src_i*saved_pump_params.dz)/C0,
        pump_record_t, pump_record_source, left=0, right=0,
    )
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    axes[0].plot(t_ax*1e3, waveform_padded/np.max(abs(waveform_padded)), label="Probe at source")
    axes[0].plot(t_ax*1e3, probe_front_field/np.max(abs(probe_front_field)), label="Incident probe at front (vacuum delay)")
    axes[0].plot(t_ax*1e3, pump_front_field/np.max(abs(pump_record_source)), label="Incident pump at front (vacuum delay)")
    axes[0].axvline(probe_incident_peak_front_ps*1e3, color="C1", ls=":")
    axes[0].axvline(probe_incident_peak_front_ps*1e3-REPRESENTATIVE_DELAY_FS, color="C2", ls=":")
    axes[0].set(ylabel="Normalized incident field", title=f"Incident-peak delay = {REPRESENTATIVE_DELAY_FS:+.3f} fs (positive = pump first)")
    axes[0].legend()
    for j, z in enumerate(diagnostic_z):
        axes[1].plot(t_ax*1e3, representative_omega_D[:, j], label=f"depth {(z-sample_front_pos)*1e-4:.3f} um", alpha=0.35+0.65*j/max(1, len(diagnostic_z)-1), color="black")
    axes[1].axhline(del_omega_D, color="k", lw=0.8, ls="--", label="Global maximum increase")
    axes[1].set(xlabel="Absolute probe-simulation time (fs)", ylabel="omega_D (rad/ps)")
    axes[1].legend(ncol=2)
    diagnostic_end_ps = max(source_waveform_end_time, float(np.nanmax(pump_injection_hi_ps))
                            + pump_time_shift_ps(REPRESENTATIVE_DELAY_FS)) + 10e-3
    axes[1].set_xlim(0, min(t_ax[-1], max(20e-3, diagnostic_end_ps))*1e3)
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.savefig(base_dir / "incident_timing_and_injection_profiles.png", dpi=180)
    plt.show()
    np.savez_compressed(base_dir / "representative_injection_profiles.npz",
                        t_ps=t_ax, z_A=diagnostic_z, omega_D=representative_omega_D,
                        delay_fs=REPRESENTATIVE_DELAY_FS)

    # The final overlap/delay-window plot is made after the unpumped timing reference.


The following diagnostic shows the simulation geometry.


In [11]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Simulation geometry diagnostic
    import numpy as np
    import matplotlib.pyplot as plt

    # -----------------------------------------------------------------------------
    # Plot the spatial configuration of the FDTD simulation
    # -----------------------------------------------------------------------------

    # Conversion factor
    angstrom_to_um = 1.0e-4

    # Full simulation-domain boundaries
    domain_left = 0.0
    domain_right = Nz * dz

    # Actual grid-aligned positions
    source_position = src_i * dz
    reflection_probe_position = probe_r_i * dz
    transmission_probe_position = probe_t_i * dz

    metal_left = metal_i0 * dz
    metal_right = metal_i1 * dz

    # Convert positions from angstroms to micrometres
    domain_left_um = domain_left * angstrom_to_um
    domain_right_um = domain_right * angstrom_to_um

    source_position_um = source_position * angstrom_to_um
    reflection_probe_position_um = reflection_probe_position * angstrom_to_um
    transmission_probe_position_um = transmission_probe_position * angstrom_to_um

    metal_left_um = metal_left * angstrom_to_um
    metal_right_um = metal_right * angstrom_to_um

    # -----------------------------------------------------------------------------
    # Create plot
    # -----------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(13, 3.5))

    # Draw the full simulation region
    ax.hlines(
        y=0,
        xmin=domain_left_um,
        xmax=domain_right_um,
        linewidth=3,
        label="Simulation domain",
    )

    # Shade the fused-silica sample
    ax.axvspan(
        metal_left_um,
        metal_right_um,
        alpha=0.45,
        label=f"Fused-silica sample ({thickness*1.0e-4:.2f} μm)",
    )

    # Source
    ax.axvline(
        source_position_um,
        linestyle="--",
        linewidth=2,
        label=f"Source: {source_position_um:.3f} μm",
    )

    # Reflection probe
    ax.axvline(
        reflection_probe_position_um,
        linestyle="-.",
        linewidth=2,
        label=f"Reflection probe: {reflection_probe_position_um:.3f} μm",
    )

    # Transmission probe
    ax.axvline(
        transmission_probe_position_um,
        linestyle=":",
        linewidth=2.5,
        label=f"Transmission probe: {transmission_probe_position_um:.3f} μm",
    )

    # Add markers along the simulation line
    ax.scatter(source_position_um, 0, marker="*", s=180, zorder=5)
    ax.scatter(reflection_probe_position_um, 0, marker="o", s=80, zorder=5)
    ax.scatter(transmission_probe_position_um, 0, marker="s", s=80, zorder=5)



    # Formatting
    ax.set_xlim(domain_left_um, domain_right_um)
    ax.set_ylim(-0.75, 0.9)

    ax.set_xlabel("Position, $z$ (μm)")
    ax.set_yticks([])
    ax.set_title("Spatial Configuration of the 1D FDTD Simulation")

    ax.grid(axis="x", alpha=0.25)
    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
        ncol=3,
        frameon=False,
    )

    plt.tight_layout()
    fig.savefig(base_dir / "simulation_geometry.png", dpi=180)
    plt.show()

    # -----------------------------------------------------------------------------
    # Print exact grid-aligned positions
    # -----------------------------------------------------------------------------
    print(f"Simulation domain:       {domain_left_um:.4f} to {domain_right_um:.4f} μm")
    print(f"Source position:         {source_position_um:.4f} μm")
    print(f"Reflection probe:        {reflection_probe_position_um:.4f} μm")
    print(f"Transmission probe:      {transmission_probe_position_um:.4f} μm")
    print(f"Sample region:            {metal_left_um:.4f} to {metal_right_um:.4f} μm")
    print(f"Discretized sample width: {(metal_right - metal_left):.1f} Å")

## Run and store the references and delay scan

The next cells run two controls (unpumped and the final, spatially varying injected plateau), then the complete delay scan. Interior probes in the unpumped control refine the scan bounds to include reflected probe pulses; the reported final run count includes the outer echo-sensitive intervals. Each run saves `data.npz` with transmitted/reflected traces and `info.json` with exact delay, time shift, parameters, and a pointer to the shared injection cache. The shared cache, its axes, and each run's shift and scale reconstruct the full `omega_D(t,z)` without duplicating gigabytes per delay. Keep `shared_injection/` with the run folders. These compact files use the reader below rather than `load_simulation_run`, which expects all material arrays in every archive.

The final runner only builds a cache/scan in a new case directory, and refuses incomplete or incompatible saved scans. Completed compatible scans are reused for analysis. A manifest records the planned delays and completed runs; analysis refuses a partial scan. Only the current manifest's indexed run folders are used, avoiding collisions between rounded delay labels.

In [12]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Compact result persistence and reusable material construction
    background_kwargs = dict(
        background_del_eps=saved_pump_metadata["background_del_eps"],
        background_lambda0_um=saved_pump_metadata["background_lambda0_um"],
        background_gamma=saved_pump_metadata["background_gamma_per_ps"],
    )
    common_material = dict(
        gamma_D=gamma_D_constant, omega_L=np.zeros(Nt), gamma_L=np.zeros(Nt),
        del_eps=np.zeros(Nt), eps_inf=np.ones(Nt),
    )
    scan_configuration = {
        "delay_definition": "probe incident abs(E) peak minus pump incident abs(E) peak at the sample front; positive=pump first",
        "omega_D_model": f"omega_D0 + del_omega_D * sqrt(integral pump E^{PUMP_FIELD_EXPONENT} dt) / max_z sqrt(integral pump E^{PUMP_FIELD_EXPONENT} dt at record end)",
        "pump_field_exponent": int(PUMP_FIELD_EXPONENT),
        "injection_outer_power": 0.5,
        "omega_D0_rad_per_ps": omega_D0, "del_omega_D_rad_per_ps": del_omega_D,
        "gamma_D_per_ps": gamma_D0, "interpolation": SPATIAL_INTERPOLATION,
        "interpolation_validation": interpolation_errors,
        "response_normalization": response_normalization,
        "response_normalization_units": f"code_field^{PUMP_FIELD_EXPONENT//2} sqrt(ps)",
        "cumulative_cache_max_absolute_fraction_error": cache_time_error,
        "pump_record_final_5fs_max_local_response_fraction": record_tail_fraction,
        "pump_run_directory": str(PUMP_RUN_DIR), "pump_metadata": saved_pump_metadata,
        "pump_common_envelope_phase_rad": float(MATCHING_PUMP_PHASE_RAD),
        "pump_common_envelope_phase_label": MATCHING_PUMP_PHASE_LABEL,
        "pump_envelope_center_source_fs": saved_pump_metadata["envelope_center_fs"],
        "pump_incident_abs_peak_source_fs": saved_pump_metadata["incident_abs_peak_source_fs"],
        "probe_incident_peak_front_ps": float(probe_incident_peak_front_ps),
        "pump_incident_peak_front_ps": float(pump_incident_peak_front_ps),
        "experimental_trace_index": corrected_source_metadata["reference_trace_index"],
        "probe_source_provenance": probe_source_provenance,
        "probe_source_peak_E0": E0,
        "probe_source_baseline_subtracted_in_notebook": False,
        "probe_source_time_convention": "corrected group-delay-removed axis shifted to start at zero, then SOURCE_PREPAD_FS added",
        "probe_source_correction_metadata": corrected_source_metadata,
        "energy_tail_fraction": ENERGY_TAIL_FRACTION, "scan_margin_fs": SCAN_MARGIN_FS,
        "probe_occupancy_tail_fraction": PROBE_OCCUPANCY_TAIL_FRACTION,
        "probe_timing_every_cells": PROBE_TIMING_EVERY_CELLS,
        "dense_main_overlap_samples": N_DELAYS, "echo_delay_step_fs": ECHO_DELAY_STEP_FS,
        "minimum_post_record_margin_fs": POST_RECORD_MARGIN_FS,
        "actual_record_margin_fs": record_margin_ps*1e3,
        "minimum_internal_round_trips": MIN_INTERNAL_ROUND_TRIPS,
        "minimum_Drude_damping_times": MIN_DRUDE_DAMPING_TIMES,
        "probe_group_index_bounds": [ng_min, ng_max],
        "tail_check_fs": TAIL_CHECK_FS, "tail_energy_tolerance": TAIL_ENERGY_TOLERANCE,
        "fdtd_params": asdict(simulation_params_withfilm), "pulse_params": asdict(pulse),
        "material_array_axes": ["simulation_time", "material_cell"],
        "shared_profile_quantity": f"globally normalized sqrt(integral pump E^{PUMP_FIELD_EXPONENT} dt)",
        "shared_profile": f"shared_injection/normalized_sqrt_cumulative_E{PUMP_FIELD_EXPONENT}.npy",
        "shared_axes": "shared_injection/axes_and_timing.npz",
    }
    (base_dir / "scan_configuration.json").write_text(json.dumps(scan_configuration, indent=2))


    def save_probe_run(folder, result, *, delay_fs=None, state="pump_driven"):
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        E_t, E_r, snaps_t, snaps_E = result
        if not np.all(np.isfinite(E_t)) or not np.all(np.isfinite(E_r)):
            raise RuntimeError("Probe simulation produced non-finite fields.")
        total = float(trapezoid(E_t**2, t_ax))
        tail_start = max(0, np.searchsorted(t_ax, t_ax[-1]-TAIL_CHECK_FS*1e-3)-1)
        tail_fraction = float(trapezoid(E_t[tail_start:]**2, t_ax[tail_start:])/total) if total > 0 else 0.0
        np.savez_compressed(folder / "data.npz", E_t=E_t, E_r=E_r, t_ps=t_ax,
                            snaps_t=snaps_t, snaps_E=snaps_E)
        info = {
            "fdtd_params": asdict(simulation_params_withfilm), "pulse_params": asdict(pulse),
            "material_state": state, "delay_fs": None if delay_fs is None else float(delay_fs),
            "pump_time_shift_ps": None if delay_fs is None else float(pump_time_shift_ps(delay_fs)),
            "omega_D0_rad_per_ps": omega_D0, "del_omega_D_rad_per_ps": del_omega_D,
            "gamma_D_per_ps": gamma_D0,
            "pump_field_exponent": int(PUMP_FIELD_EXPONENT),
            "pump_common_envelope_phase_rad": float(MATCHING_PUMP_PHASE_RAD),
            "pump_common_envelope_phase_label": MATCHING_PUMP_PHASE_LABEL,
            "scan_configuration": "../scan_configuration.json",
            "shared_profile_quantity": f"globally normalized sqrt(integral pump E^{PUMP_FIELD_EXPONENT} dt)",
            "shared_profile": f"../shared_injection/normalized_sqrt_cumulative_E{PUMP_FIELD_EXPONENT}.npy",
            "shared_axes": "../shared_injection/axes_and_timing.npz",
            "transmitted_energy_code_field2_ps": total,
            "last_window_transmitted_energy_fraction": tail_fraction,
            "tail_check_fs": TAIL_CHECK_FS,
        }
        (folder / "info.json").write_text(json.dumps(info, indent=2))
        return info


    def simulate_probe(omega_D, material_probe_indices=None):
        material = MaterialParamsTD(omega_D=omega_D, **common_material)
        return run_simulation(simulation_params_withfilm, pulse, material,
                              method="sio2_background", explicit_pulse_waveform=waveform_padded,
                              material_probe_indices=material_probe_indices, **background_kwargs)


    def load_probe_run(folder):
        folder = Path(folder)
        with (folder / "info.json").open() as stream:
            info = json.load(stream)
        with np.load(folder / "data.npz") as archive:
            data = {name: archive[name] for name in ("E_t", "E_r", "t_ps")}
        if data["E_t"].shape != data["t_ps"].shape:
            raise ValueError("Saved trace and time-axis dimensions disagree.")
        return data, info


    def reconstruct_saved_omega_D(folder, out=None):
        """Reconstruct material arrays using only saved artifacts; pass a memmap as out."""
        folder = Path(folder)
        with (folder / "info.json").open() as stream:
            info = json.load(stream)
        profile = np.load(folder / info["shared_profile"], mmap_mode="r")
        with np.load(folder / info["shared_axes"]) as archive:
            times = archive["t_ps"]
        p = info["fdtd_params"]
        sample_times = np.arange(p["Nt"])*p["dt"]
        if info["material_state"] == "unpumped":
            return np.full(p["Nt"], info["omega_D0_rad_per_ps"])
        if info["material_state"] == "fully_injected":
            return np.broadcast_to(info["omega_D0_rad_per_ps"]+info["del_omega_D_rad_per_ps"]*profile[-1], (p["Nt"], profile.shape[1]))
        return resample_cumulative(times, profile, sample_times-info["pump_time_shift_ps"], out=out,
                                   scale=info["del_omega_D_rad_per_ps"], baseline=info["omega_D0_rad_per_ps"])

In [13]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Reference simulations: no carriers and the saturated spatial injection profile.
    reference_probe_indices = np.unique(np.r_[
        np.arange(metal_i0, metal_i1, PROBE_TIMING_EVERY_CELLS), metal_i1-1,
    ])
    reference_with_probes = simulate_probe(np.full(Nt, omega_D0), material_probe_indices=reference_probe_indices)
    reference_result, reference_probes = reference_with_probes[:4], reference_with_probes[4]
    del reference_with_probes
    unpumped_info = save_probe_run(base_dir / "unpumped_reference", reference_result, state="unpumped")
    fully_injected_profile = omega_D0 + del_omega_D*injection_fraction[-1]
    fully_injected_result = simulate_probe(np.broadcast_to(fully_injected_profile, (Nt, metal_cells)))
    fully_injected_info = save_probe_run(base_dir / "fully_injected_reference", fully_injected_result, state="fully_injected")
    for label, info in (("unpumped", unpumped_info), ("fully injected", fully_injected_info)):
        print(f"{label}: last {TAIL_CHECK_FS:g} fs energy fraction = {info['last_window_transmitted_energy_fraction']:.3g}")
        if info["last_window_transmitted_energy_fraction"] > TAIL_ENERGY_TOLERANCE:
            warnings.warn(f"The {label} reference still has a transmitted tail; increase POST_RECORD_MARGIN_FS and rerun.")

    # Refine the occupation interval using the full unpumped local fields, including echoes.
    reference_interval_ps = np.empty((reference_probe_indices.size, 2))
    for j in range(reference_probe_indices.size):
        cumulative = cumulative_trapezoid(reference_probes["E"][:, j]**2, t_ax, initial=0)
        reference_interval_ps[j] = integral_quantiles(
            t_ax, cumulative, (PROBE_OCCUPANCY_TAIL_FRACTION, 1-PROBE_OCCUPANCY_TAIL_FRACTION),
        )
    if not np.all(np.isfinite(reference_interval_ps)):
        raise ValueError("At least one reference timing probe recorded zero energy; check the simulation duration.")
    probe_lo_from_peak = np.interp(material_z_A, reference_probes["z"], reference_interval_ps[:, 0])-probe_incident_peak_front_ps
    probe_hi_from_peak = np.interp(material_z_A, reference_probes["z"], reference_interval_ps[:, 1])-probe_incident_peak_front_ps
    delay_min_fs = float(np.min((pump_lo_from_peak-probe_hi_from_peak)[active_cells])*1e3-SCAN_MARGIN_FS)
    delay_max_fs = float(np.max((pump_hi_from_peak-probe_lo_from_peak)[active_cells])*1e3+SCAN_MARGIN_FS)
    core_delays = np.linspace(main_delay_min_fs, main_delay_max_fs, N_DELAYS)
    outer_early = np.arange(delay_min_fs, min(main_delay_min_fs, delay_max_fs), ECHO_DELAY_STEP_FS)
    outer_late = np.arange(max(main_delay_max_fs, delay_min_fs)+ECHO_DELAY_STEP_FS, delay_max_fs, ECHO_DELAY_STEP_FS)
    delays_fs = np.unique(np.r_[core_delays, outer_early, outer_late, delay_min_fs, delay_max_fs, 0.0])
    delays_fs = delays_fs[(delays_fs >= delay_min_fs) & (delays_fs <= delay_max_fs)]
    np.savez_compressed(shared_dir / "reference_probe_occupation.npz",
                        z_A=reference_probes["z"], energy_quantile_interval_ps=reference_interval_ps,
                        local_probe_lo_from_peak_ps=probe_lo_from_peak, local_probe_hi_from_peak_ps=probe_hi_from_peak)
    del reference_probes
    scan_configuration.update({
        "delay_min_fs": delay_min_fs, "delay_max_fs": delay_max_fs,
        "delay_fs": delays_fs.tolist(), "main_overlap_bounds_fs": [main_delay_min_fs, main_delay_max_fs],
        "delay_bounds_method": f"local pump sqrt(integral E{PUMP_FIELD_EXPONENT}) response fractions and measured unpumped probe E2 quantile overlap, including recorded echoes",
    })
    (base_dir / "scan_configuration.json").write_text(json.dumps(scan_configuration, indent=2))
    print(f"Final incident-peak delay scan: {delay_min_fs:.3f} to {delay_max_fs:.3f} fs; {len(delays_fs)} runs")
    print(f"Dense main-pulse interval: {main_delay_min_fs:.3f} to {main_delay_max_fs:.3f} fs")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(local_depth*1e-4, (pump_lo_from_peak-probe_hi_from_peak)*1e3,
                    (pump_hi_from_peak-probe_lo_from_peak)*1e3, alpha=0.3, label="Local overlap including probe echoes")
    ax.axhline(delay_min_fs, color="k", ls="--", label="Final scan bounds including margins")
    ax.axhline(delay_max_fs, color="k", ls="--")
    ax.axhspan(main_delay_min_fs, main_delay_max_fs, color="C1", alpha=0.15, label="Dense main-pulse interval")
    ax.set(xlabel="Depth into sample (um)", ylabel="Probe-minus-pump incident-peak delay (fs)")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(base_dir / "delay_scan_window.png", dpi=180)
    plt.show()

    # Existing static-trace comparison, aligned using the calculated transit time.
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t_ax*1e3, reference_result[0]/np.max(abs(reference_result[0])), label="Unpumped transmitted probe")
    ax.plot((t_ax+source_to_transmission_time)*1e3, waveform_padded/np.max(abs(waveform_padded)), label="Incident probe shifted by estimated transit")
    ax.set(xlabel="Simulation time (fs)", ylabel="Normalized field")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(base_dir / "static_transmitted_waveform.png", dpi=180)
    plt.show()

In [14]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Scan relative incident-peak delays. This cell runs the dense main interval plus the outer echo intervals.
    scan_rows = [
        {"index": i, "delay_fs": float(delay), "folder": f"delay_{i:04d}_{delay:+.6f}_fs"}
        for i, delay in enumerate(delays_fs)
    ]
    manifest = {"complete": False, "runs": scan_rows, "completed_indices": [],
                "pump_field_exponent": int(PUMP_FIELD_EXPONENT),
                "pump_common_envelope_phase_rad": float(MATCHING_PUMP_PHASE_RAD)}
    manifest_path = base_dir / "scan_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    work_path = base_dir / "_omega_D_work.npy"
    omega_D_work = np.lib.format.open_memmap(work_path, mode="w+", dtype=np.float64, shape=(Nt, metal_cells))
    tail_failures = []
    try:
        for row in tqdm(scan_rows, desc="Pump-driven probe delay scan", unit="run"):
            delay = row["delay_fs"]
            resample_cumulative(injection_time_ps, injection_fraction, t_ax-pump_time_shift_ps(delay),
                                out=omega_D_work, scale=del_omega_D, baseline=omega_D0)
            result = simulate_probe(omega_D_work)
            info = save_probe_run(base_dir / row["folder"], result, delay_fs=delay)
            if info["last_window_transmitted_energy_fraction"] > TAIL_ENERGY_TOLERANCE:
                tail_failures.append(delay)
            manifest["completed_indices"].append(row["index"])
            manifest_path.write_text(json.dumps(manifest, indent=2))
        manifest["complete"] = True
        manifest["delays_requiring_tail_check_fs"] = tail_failures
        manifest_path.write_text(json.dumps(manifest, indent=2))
    finally:
        omega_D_work.flush()
        del omega_D_work
        work_path.unlink(missing_ok=True)
    if tail_failures:
        warnings.warn(f"{len(tail_failures)} delays exceed the tail tolerance; increase POST_RECORD_MARGIN_FS and check convergence.")
    print(f"Saved {len(scan_rows)} delay runs and shared injection data to {base_dir}")

# Analysis

Integrated transmitted energy is normalized to the unpumped reference. The incident waveform/vector-potential overlays use the same **incident peak** time origin as the delay axis; they are timing references, not a material-arrival or energy-centroid convention. A separate plot above retains the unpumped transmitted waveform comparison. The two scan endpoints are also compared with the unpumped and fully injected controls to assess the chosen delay range.

This standalone diagnostic integrates the full saved record with the trapezoidal rule. Its plot shows **pump-minus-probe** delay (`-delay_fs`); saved delays retain the probe-minus-pump convention. The final batch comparison instead uses the experimental notebook's `sum(E**2)*dt` on the aligned experimental window.


In [15]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Load the compact scan outputs and compare integrated transmitted energy.
    with (base_dir / "scan_manifest.json").open() as stream:
        saved_manifest = json.load(stream)
    if not saved_manifest["complete"] or len(saved_manifest["completed_indices"]) != len(saved_manifest["runs"]):
        raise RuntimeError("The delay scan is incomplete; finish it before running complete-scan analysis.")
    unpumped, _ = load_probe_run(base_dir / "unpumped_reference")
    fully_injected, _ = load_probe_run(base_dir / "fully_injected_reference")
    energy_static = trapezoid(unpumped["E_t"]**2, unpumped["t_ps"]*UNIT_T)
    if energy_static <= 0:
        raise ValueError("Unpumped transmitted energy is zero.")
    energy_fully = trapezoid(fully_injected["E_t"]**2, fully_injected["t_ps"]*UNIT_T)
    analysis_delays_fs = np.array([row["delay_fs"] for row in saved_manifest["runs"]])
    energy_arr = np.empty(analysis_delays_fs.size)
    tail_arr = np.empty_like(energy_arr)
    for i, row in enumerate(tqdm(saved_manifest["runs"], desc="Integrating transmitted energy")):
        data, info = load_probe_run(base_dir / row["folder"])
        energy_arr[i] = trapezoid(data["E_t"]**2, data["t_ps"]*UNIT_T)
        tail_arr[i] = info["last_window_transmitted_energy_fraction"]
    integrated_energy_ratio = energy_arr/energy_static
    vector_potential_incident = -cumulative_trapezoid(reference_field, waveform_time_rel_ps*UNIT_T, initial=0)

    def normalized(values):
        maximum = np.max(abs(values))
        return values/maximum if maximum > 0 else np.zeros_like(values)

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.plot(-analysis_delays_fs, (integrated_energy_ratio-1)*100, label="Transmitted-energy change")
    ax1.scatter(-analysis_delays_fs, (integrated_energy_ratio-1)*100)

    ax1.axhline(0, color="0.4", lw=1, label="Unpumped limit")
    ax1.axhline((energy_fully/energy_static-1)*100, color="C3", ls=":", label="Fully injected spatial-profile limit")
    ax1.set(xlabel="Pump-minus-probe incident-peak delay (fs; positive = probe first)",
            ylabel="Change in transmitted energy (%)", title=f"Probe transmission: sqrt(integral E^{PUMP_FIELD_EXPONENT}), pump CEP = 0")
    ax1.grid(alpha=0.3)
    ax2 = ax1.twinx()
    incident_peak_time_fs = (waveform_time_rel_ps-probe_incident_peak_rel_ps)*1e3
    # Pump peak position in the fixed probe frame is -delay, hence the minus sign.
    ax2.plot(incident_peak_time_fs, normalized(reference_field), color="black", alpha=0.45, ls="--", label="Incident probe sampled at pump-peak delay")
    ax2.plot(incident_peak_time_fs, normalized(vector_potential_incident), color="blue", alpha=0.45, ls="--", label="Incident vector potential (same delay mapping)")
    ax2.set_ylabel("Normalized incident reference")
    ax1.set_xlim(-analysis_delays_fs.max(), -analysis_delays_fs.min())
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc="best")
    fig.tight_layout()
    fig.savefig(base_dir / "transmitted_energy_vs_peak_delay.png", dpi=180)
    plt.show()
    np.savez_compressed(base_dir / "delay_scan_analysis.npz", delay_fs=analysis_delays_fs,
                        pump_field_exponent=PUMP_FIELD_EXPONENT,
                        transmitted_energy_ratio=integrated_energy_ratio,
                        unpumped_energy_SI_time=energy_static, fully_injected_energy_ratio=energy_fully/energy_static,
                        last_window_energy_fraction=tail_arr)
    print(f"Late-pump endpoint / unpumped energy: {integrated_energy_ratio[0]:.6g} (target 1)")
    print(f"Early-pump endpoint / unpumped energy: {integrated_energy_ratio[-1]:.6g}")
    print(f"Fully injected reference / unpumped energy: {energy_fully/energy_static:.6g}")
    print("Increase the scan margin if endpoint traces/energies have not approached their controls.")

    # Complete saved waveforms at representative delays, including late echoes.
    waveform_preview_indices = np.unique([
        0, int(np.argmin(abs(analysis_delays_fs))), len(analysis_delays_fs)-1,
    ])
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
    axes[0].plot(unpumped["t_ps"]*1e3, unpumped["E_t"], label="Unpumped reference", color="black")
    axes[0].plot(fully_injected["t_ps"]*1e3, fully_injected["E_t"], label="Fully injected control", ls=":")
    for preview_color_index, preview_index in enumerate(waveform_preview_indices, start=1):
        row = saved_manifest["runs"][preview_index]
        data, _ = load_probe_run(base_dir / row["folder"])
        if not np.array_equal(data["t_ps"], unpumped["t_ps"]):
            raise ValueError("Waveform preview and unpumped reference use different time grids.")
        label = f"Delay {row['delay_fs']:+.3f} fs"
        axes[0].plot(data["t_ps"]*1e3, data["E_t"], label=label, color=f"C{preview_color_index}", alpha=0.8)
        axes[1].plot(data["t_ps"]*1e3, data["E_t"]-unpumped["E_t"], label=label, color=f"C{preview_color_index}")
    axes[0].set(ylabel="Transmitted field (code units)", title=f"Complete transmitted waveforms: E^{PUMP_FIELD_EXPONENT}, pump CEP = 0")
    axes[1].set(xlabel="Simulation time (fs)", ylabel="Pumped − unpumped field")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=9)
    fig.savefig(base_dir / "transmitted_waveforms.png", dpi=180)
    plt.show()


# Single-lobe analysis of the simulated delay scan

This section uses **only saved simulated transmitted fields** (`E_t`), with the saved **unpumped simulation** as the reference. It reproduces the signed-peak selection and lobe integration from `experimental_data_analysis.ipynb`: select the peak nearest one fixed target time, locate that trace's own linearly interpolated bounding zeros, and integrate $E^2$ between them using the trapezoidal rule. The ratio is $U_{\mathrm{lobe,pumped}}/U_{\mathrm{lobe,unpumped}}$. No experimental waveforms are loaded or plotted.

`LOBE_TARGET_TIME_FS` is on the **saved transmitted-waveform clock**, not the incident clock or the experimental 33.5 fs clock. The default (`None`) chooses the strongest eligible positive peak of the unpumped transmitted reference once; all delays use that same target without realigning individual traces. Set an explicit time to select another lobe. The shaded previews show each trace's own integration interval, and the peak-time diagnostic reveals possible switching between neighbouring lobes.

The stored coordinate is $\Delta=t_{\mathrm{probe,peak}}-t_{\mathrm{pump,peak}}$ at the sample front: **positive saved delay means pump first**. The ratio plot displays $-\Delta$ (pump-minus-probe, positive means probe first); the peak-time diagnostic retains $\Delta$. Run these cells after Analysis, or set `LOBE_SCAN_DIR` to an existing complete scan to analyze it without rerunning FDTD. Outputs go to its `single_lobe_analysis` subdirectory.

The ratio plot also overlays the **unpumped transmitted probe field** and $A(t)=-\int_{t_0}^t E(t')\,dt'$, each scaled independently to unit maximum magnitude for display only. On the ratio plot, waveform samples are displayed at $t-t_{\rm transmitted,peak}$, where the origin is the unpumped field's largest absolute peak. The top axis shows waveform time relative to that peak; the vertical dashed line uses the same mapping for the fixed lobe target. This is a reference timing guide, not a fit or a reconstruction of the distributed injection time. The calculated energy ratios and saved delay coordinates are unchanged.


In [16]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    from pathlib import Path
    import json
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.integrate import trapezoid, cumulative_trapezoid
    from scipy.signal import find_peaks

    LOBE_SCAN_DIR = Path(base_dir) if "base_dir" in globals() else None  # or Path("/path/to/saved/scan")
    LOBE_PEAK_TYPE = "positive"
    LOBE_TARGET_TIME_FS = None           # None: strongest eligible unpumped transmitted peak
    LOBE_PROMINENCE_FRACTION = 0.0
    LOBE_MAX_PEAK_DISTANCE_FS = None      # optional safeguard against selecting a different cycle
    LOBE_PREVIEW_DELAYS_FS = (-20.0, 0.0, 20.0)  # nearest saved delays; duplicates removed
    LOBE_PREVIEW_HALF_WIDTH_FS = 15.0
    LOBE_DELAY_XLIM_FS = None             # None shows the full saved delay range
    SAVE_LOBE_ANALYSIS = True

    if LOBE_SCAN_DIR is None:
        raise ValueError("Set LOBE_SCAN_DIR to a saved scan folder, or run the earlier configuration cells.")
    LOBE_SCAN_DIR = Path(LOBE_SCAN_DIR)

In [17]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Same lobe algorithm as experimental_data_analysis.ipynb; all helper times are seconds.
    def _time_array(t):
        t = np.asarray(t, dtype=float)
        if t.ndim != 1 or t.size < 3 or not np.all(np.isfinite(t)):
            raise ValueError("t must be a finite 1D array with at least three samples.")
        if not np.all(np.diff(t) > 0.0):
            raise ValueError("t must be strictly increasing; check the input time axis.")
        return t


    def _waveform(E, t):
        if np.iscomplexobj(E):
            raise ValueError("Use the real electric-field waveform, not a complex envelope.")
        E = np.asarray(E, dtype=float)
        if E.shape != t.shape or not np.all(np.isfinite(E)):
            raise ValueError("A waveform must be finite and have the same shape as t.")
        return E


    def _signed_peaks(E, peak_type, peak_prominence_fraction):
        if peak_type not in ("positive", "negative"):
            raise ValueError("peak_type must be 'positive' or 'negative'.")
        if not np.isfinite(peak_prominence_fraction) or peak_prominence_fraction < 0:
            raise ValueError("peak_prominence_fraction must be finite and nonnegative.")
        sign = 1.0 if peak_type == "positive" else -1.0
        prominence = (peak_prominence_fraction * np.max(np.abs(E))
                      if peak_prominence_fraction > 0 else None)
        peaks, _ = find_peaks(sign * E, prominence=prominence)
        # Reject, for example, a noise-induced local maximum that is still negative.
        return peaks[sign * E[peaks] > 0.0]


    def _zero_crossings_linear(t, E):
        # No dependency on the custom lightmatter package is needed.
        idx = np.flatnonzero(((E[:-1] < 0) & (E[1:] > 0))
                             | ((E[:-1] > 0) & (E[1:] < 0)))
        # Magnitude form of linear interpolation avoids sign ambiguity.
        fraction = np.abs(E[idx]) / (np.abs(E[idx]) + np.abs(E[idx + 1]))
        crossing_times = t[idx] + fraction * (t[idx + 1] - t[idx])
        return np.unique(np.r_[crossing_times, t[E == 0.0]])


    def peak_lobe_energy(t, E, t_closest_SI, peak_type="positive", *,
                         peak_prominence_fraction=0.0, max_peak_distance_s=None,
                         return_details=False):
        """Nearest signed peak, own bounding zeros, trapezoidal integral of raw E**2.

        t and t_closest_SI are in seconds, on the saved simulation waveform time
        coordinate. No propagation-time correction or per-delay alignment is applied.
        Prominence and distance safeguards are optional; no smoothing is performed.
        """
        t = _time_array(t)
        E = _waveform(E, t)
        if not np.isfinite(t_closest_SI) or not t[0] <= t_closest_SI <= t[-1]:
            raise ValueError("t_closest_SI must be finite and within the sampled time range.")
        if max_peak_distance_s is not None:
            if not np.isfinite(max_peak_distance_s) or max_peak_distance_s < 0:
                raise ValueError("max_peak_distance_s must be finite and nonnegative.")
        peaks = _signed_peaks(E, peak_type, peak_prominence_fraction)
        if peaks.size == 0:
            raise ValueError(f"No eligible {peak_type} peaks were found.")
        peak_id = peaks[np.argmin(np.abs(t[peaks] - t_closest_SI))]
        t_peak = t[peak_id]
        if max_peak_distance_s is not None and abs(t_peak - t_closest_SI) > max_peak_distance_s:
            raise ValueError("Nearest eligible peak lies outside max_peak_distance_s.")

        zeros = _zero_crossings_linear(t, E)
        before, after = zeros[zeros < t_peak], zeros[zeros > t_peak]
        if before.size == 0 or after.size == 0:
            raise ValueError("The selected peak is not bounded by zeros on both sides.")
        t_left, t_right = before[-1], after[0]
        inside = (t > t_left) & (t < t_right)
        t_interval = np.r_[t_left, t[inside], t_right]
        E_interval = np.r_[0.0, E[inside], 0.0]
        energy = float(trapezoid(E_interval ** 2, x=t_interval))
        if not np.isfinite(energy) or energy <= 0.0:
            raise ValueError("The selected lobe has zero or nonfinite energy.")
        details = {
            "energy": energy, "peak_index": int(peak_id), "peak_time_s": float(t_peak),
            "left_zero_s": float(t_left), "right_zero_s": float(t_right),
        }
        return details if return_details else energy


    def _load_lobe_trace(folder):
        with (Path(folder) / "data.npz").open("rb") as stream, np.load(stream, allow_pickle=False) as data:
            time_s = _time_array(data["t_ps"] * 1e-12)
            field = _waveform(data["E_t"], time_s)
        with (Path(folder) / "info.json").open() as stream:
            info = json.load(stream)
        return time_s, field, info

In [18]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Stream completed saved runs; no simulation, experimental-data loading, or field rescaling.
    with (LOBE_SCAN_DIR / "scan_manifest.json").open() as stream:
        lobe_manifest = json.load(stream)
    lobe_rows = lobe_manifest["runs"]
    lobe_indices = [row["index"] for row in lobe_rows]
    if (not lobe_rows or len(set(lobe_indices)) != len(lobe_indices)
            or not lobe_manifest.get("complete", False)
            or set(lobe_manifest.get("completed_indices", [])) != set(lobe_indices)):
        raise ValueError("Single-lobe analysis requires a complete scan with unique manifest run indices.")
    lobe_delays_fs = np.array([row["delay_fs"] for row in lobe_rows], dtype=float)
    if not np.all(np.isfinite(lobe_delays_fs)) or np.unique(lobe_delays_fs).size != len(lobe_rows):
        raise ValueError("Manifest delays must be finite and unique.")
    lobe_time_s, lobe_reference_field, lobe_ref_info = _load_lobe_trace(LOBE_SCAN_DIR / "unpumped_reference")
    if lobe_ref_info.get("material_state") != "unpumped":
        raise ValueError("The lobe denominator must be the unpumped simulation.")
    lobe_candidates = _signed_peaks(lobe_reference_field, LOBE_PEAK_TYPE, LOBE_PROMINENCE_FRACTION)
    if lobe_candidates.size == 0:
        raise ValueError("No eligible signed peaks in the unpumped transmitted waveform.")
    lobe_strongest = lobe_candidates[np.argsort(abs(lobe_reference_field[lobe_candidates]))[-10:]]
    print("Strongest eligible reference-peak times (fs):", np.sort(lobe_time_s[lobe_strongest]*1e15))
    lobe_target_s = (float(lobe_time_s[lobe_candidates[np.argmax(abs(lobe_reference_field[lobe_candidates]))]])
                     if LOBE_TARGET_TIME_FS is None else float(LOBE_TARGET_TIME_FS)*1e-15)
    lobe_options = dict(peak_type=LOBE_PEAK_TYPE, peak_prominence_fraction=LOBE_PROMINENCE_FRACTION,
                        max_peak_distance_s=None if LOBE_MAX_PEAK_DISTANCE_FS is None else LOBE_MAX_PEAK_DISTANCE_FS*1e-15,
                        return_details=True)
    lobe_reference = peak_lobe_energy(lobe_time_s, lobe_reference_field, lobe_target_s, **lobe_options)
    lobe_preview_requests = np.asarray(LOBE_PREVIEW_DELAYS_FS, dtype=float)
    if (lobe_preview_requests.ndim != 1 or not lobe_preview_requests.size
            or not np.all(np.isfinite(lobe_preview_requests))):
        raise ValueError("LOBE_PREVIEW_DELAYS_FS must be a nonempty finite 1D sequence.")
    if not np.isfinite(LOBE_PREVIEW_HALF_WIDTH_FS) or LOBE_PREVIEW_HALF_WIDTH_FS <= 0:
        raise ValueError("LOBE_PREVIEW_HALF_WIDTH_FS must be positive and finite.")
    lobe_preview_indices = np.unique([np.argmin(abs(lobe_delays_fs-delay)) for delay in lobe_preview_requests])
    lobe_preview_fields = {}
    lobe_scan = {key: np.empty(len(lobe_rows)) for key in
                 ("energy_modified", "modified_peak_time_s", "modified_left_zero_s", "modified_right_zero_s")}
    for j, row in enumerate(lobe_rows):
        run_time, run_field, run_info = _load_lobe_trace(LOBE_SCAN_DIR / row["folder"])
        if (not np.array_equal(run_time, lobe_time_s)
                or run_info.get("material_state") != "pump_driven"
                or not np.isclose(run_info["delay_fs"], row["delay_fs"], rtol=0, atol=1e-9)
                or run_info["fdtd_params"] != lobe_ref_info["fdtd_params"]
                or run_info["pulse_params"] != lobe_ref_info["pulse_params"]):
            raise ValueError(f"Saved run/reference/manifest mismatch: {row['folder']}")
        try:
            details = peak_lobe_energy(run_time, run_field, lobe_target_s, **lobe_options)
        except ValueError as exc:
            raise ValueError(f"Lobe selection failed at delay {row['delay_fs']:g} fs ({row['folder']}): {exc}") from exc
        lobe_scan["energy_modified"][j] = details["energy"]
        for key in ("peak_time_s", "left_zero_s", "right_zero_s"):
            lobe_scan[f"modified_{key}"][j] = details[key]
        if j in lobe_preview_indices:
            lobe_preview_fields[j] = run_field
    lobe_scan["energy_reference"] = np.full(len(lobe_rows), lobe_reference["energy"])
    for key in ("peak_time_s", "left_zero_s", "right_zero_s"):
        lobe_scan[f"reference_{key}"] = np.full(len(lobe_rows), lobe_reference[key])
    lobe_scan["ratio"] = lobe_scan["energy_modified"] / lobe_scan["energy_reference"]
    lobe_scan["percent_change"] = 100*(lobe_scan["ratio"]-1)
    lobe_scan["delay_fs"] = lobe_delays_fs.copy()
    lobe_scan["manifest_indices"] = np.array(lobe_indices)
    print(f"Analyzed {len(lobe_rows)} simulated traces; fixed {LOBE_PEAK_TYPE} target = {lobe_target_s*1e15:.6f} fs.")

In [19]:
# Notebook stage: executed by the final exponent runner.
if not MATCHING_RUN_IN_FINAL_CELL:
    # Shaded reference and pumped lobes at representative saved delays.
    lobe_preview_figure, lobe_preview_axes = plt.subplots(
        len(lobe_preview_indices), 1, figsize=(12, 3.8*len(lobe_preview_indices)), squeeze=False,
        constrained_layout=True,
    )
    for ax, j in zip(lobe_preview_axes[:, 0], lobe_preview_indices):
        for field, prefix, color, style, label in (
            (lobe_reference_field, "reference", "C0", "-", "Unpumped transmitted reference"),
            (lobe_preview_fields[j], "modified", "C1", "--", "Pumped transmitted waveform"),
        ):
            left, right = (lobe_scan[f"{prefix}_{side}_zero_s"][j] for side in ("left", "right"))
            inside = (lobe_time_s > left) & (lobe_time_s < right)
            ax.plot(lobe_time_s*1e15, field, color=color, ls=style, label=label)
            ax.fill_between(np.r_[left, lobe_time_s[inside], right]*1e15,
                            np.r_[0.0, field[inside], 0.0], 0, color=color, alpha=0.25,
                            label=f"{prefix.capitalize()} selected lobe")
            ax.axvline(left*1e15, color=color, ls=":", alpha=0.6)
            ax.axvline(right*1e15, color=color, ls=":", alpha=0.6)
        ax.axvline(lobe_target_s*1e15, color="0.3", ls="-.", label="Fixed target time")
        ax.set(xlim=(lobe_target_s*1e15-LOBE_PREVIEW_HALF_WIDTH_FS, lobe_target_s*1e15+LOBE_PREVIEW_HALF_WIDTH_FS),
               xlabel="Saved simulation waveform time (fs)", ylabel="Electric field (saved units)",
               title=f"Pump–probe delay {lobe_delays_fs[j]:+.3f} fs; lobe energy ratio {lobe_scan['ratio'][j]:.6f}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc="upper right")
    plt.show()

    lobe_order = np.argsort(lobe_delays_fs)
    lobe_ratio_figure, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ax.plot(-lobe_delays_fs[lobe_order], lobe_scan["ratio"][lobe_order], ".-", label="Simulated lobe energy ratio")
    ax.axhline(1, color="0.4", ls=":", label="Unpumped limit")
    ax.set(xlabel="Pump-minus-probe incident-peak delay (fs; positive = probe first)",
           ylabel="Lobe energy ratio (pumped / unpumped)",
           title=f"Fixed {LOBE_PEAK_TYPE} lobe near simulation time {lobe_target_s*1e15:.3f} fs")

    # Display the unpumped transmitted probe, as in the experimental lobe plot.
    # Integrate on increasing physical time BEFORE reversing its display coordinate.
    lobe_vector_potential = -cumulative_trapezoid(lobe_reference_field, lobe_time_s, initial=0.0)
    lobe_overlay_peak_s = float(lobe_time_s[np.argmax(abs(lobe_reference_field))])
    # A later pump corresponds to a negative probe-minus-pump delay. This is a
    # transmitted-peak-frame timing guide, not an inferred material-injection time.
    lobe_overlay_delay_fs = (lobe_overlay_peak_s-lobe_time_s)*1e15
    lobe_target_display_fs = (lobe_overlay_peak_s-lobe_target_s)*1e15
    ax.axvline(-lobe_target_display_fs, color="C4", lw=1.2, ls="--",
               label="Selected-lobe target (transmitted-peak frame)")
    lobe_overlay_ax = ax.twinx()
    for values, color, style, label in (
        (lobe_reference_field, "black", "--", "Unpumped probe E (display-scaled)"),
        (lobe_vector_potential, "C3", "-.", "Unpumped probe A = −∫E dt (display-scaled)"),
    ):
        scale = np.max(abs(values))
        display_values = values/scale if scale > 0 else np.zeros_like(values)
        lobe_overlay_ax.plot(-lobe_overlay_delay_fs, display_values, color=color,
                             ls=style, lw=1.2, alpha=0.6, label=label)
    lobe_overlay_ax.set_ylabel("Probe overlays (each scaled to unit maximum magnitude)")
    # The top axis labels the displayed waveform clock. Both curves use -delay;
    # the saved delay coordinates are unchanged and no fitting shift is applied.
    lobe_waveform_axis = ax.secondary_xaxis("top", functions=(lambda x: x, lambda x: x))
    lobe_waveform_axis.set_xlabel("Overlay waveform time relative to transmitted probe peak (fs)")
    if LOBE_DELAY_XLIM_FS is not None:
        ax.set_xlim(LOBE_DELAY_XLIM_FS)
    else:
        delay_span = np.ptp(lobe_delays_fs)
        margin = 0.02*delay_span if delay_span > 0 else 1.0
        ax.set_xlim(-lobe_delays_fs.max()-margin, -lobe_delays_fs.min()+margin)
    ax.grid(alpha=0.3)
    lines, labels = ax.get_legend_handles_labels()
    overlay_lines, overlay_labels = lobe_overlay_ax.get_legend_handles_labels()
    ax.legend(lines+overlay_lines, labels+overlay_labels, fontsize=9, loc="lower left")
    plt.show()

    # A nearest-peak selection can switch lobes; expose the selected times without rematching.
    lobe_timing_figure, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
    ax.plot(lobe_delays_fs[lobe_order], lobe_scan["modified_peak_time_s"][lobe_order]*1e15,
            ".-", label="Pumped selected peak")
    ax.axhline(lobe_reference["peak_time_s"]*1e15, color="C1", ls=":", label="Unpumped selected peak")
    ax.axhline(lobe_target_s*1e15, color="0.3", ls="--", label="Fixed target")
    ax.set(xlabel="Probe-minus-pump incident-peak delay (fs; positive = pump first)",
           ylabel="Selected simulation peak time (fs)")
    if LOBE_DELAY_XLIM_FS is not None:
        ax.set_xlim(LOBE_DELAY_XLIM_FS)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

    if SAVE_LOBE_ANALYSIS:
        lobe_output_dir = LOBE_SCAN_DIR / "single_lobe_analysis"
        lobe_output_dir.mkdir(parents=True, exist_ok=True)
        for filename, figure in (("selected_lobe_previews.png", lobe_preview_figure),
                                 ("lobe_energy_ratio_vs_delay.png", lobe_ratio_figure),
                                 ("selected_peak_time_vs_delay.png", lobe_timing_figure)):
            figure.savefig(lobe_output_dir / filename, dpi=180)
        np.savez_compressed(lobe_output_dir / "lobe_energy_analysis.npz", **lobe_scan)
        lobe_metadata = dict(scan_directory=str(LOBE_SCAN_DIR.resolve()), peak_type=LOBE_PEAK_TYPE,
                             target_time_fs=lobe_target_s*1e15, requested_target_time_fs=LOBE_TARGET_TIME_FS,
                             overlay_reference_peak_time_fs=lobe_overlay_peak_s*1e15,
                             overlay_target_delay_fs=-lobe_target_display_fs,
                             overlay_coordinate="waveform time minus transmitted reference abs(E) peak time; display timing guide only",
                             vector_potential="A(t) = -cumulative_trapezoid(E, t, initial=0); time in seconds; independently display-scaled",
                             peak_prominence_fraction=LOBE_PROMINENCE_FRACTION,
                             max_peak_distance_fs=LOBE_MAX_PEAK_DISTANCE_FS,
                             preview_delays_fs=lobe_delays_fs[lobe_preview_indices].tolist(),
                             delay_definition="probe minus pump incident-peak time at sample front; positive = pump first",
                             integration="trapezoidal integral of E**2 between each trace's own interpolated zeros; time in seconds",
                             normalization="saved unpumped transmitted lobe", run_folders=[row["folder"] for row in lobe_rows])
        (lobe_output_dir / "lobe_analysis_metadata.json").write_text(json.dumps(lobe_metadata, indent=2))
        print(f"Saved simulation-only lobe figures and numerical results to {lobe_output_dir}")

In [20]:
# Run the corrected exponent variation (long-running; execute only when ready).
# Restart Kernel and Run All reuses one CEP=0 pump and creates one cache per exponent.
from pathlib import Path
import ast
import gc
import hashlib
import json
import shutil
import textwrap
from unittest.mock import patch
import matplotlib.pyplot as plt
import numpy as np

matching_notebook_path = next(
    (candidate / "notebooks/project_2/simulate_experimental_setup.ipynb"
     for root in (Path.cwd(), *Path.cwd().parents)
     for candidate in (root, root / "lightmatter")
     if (candidate / "notebooks/project_2/simulate_experimental_setup.ipynb").is_file()),
    None,
)
if matching_notebook_path is None:
    raise FileNotFoundError("Run this cell from inside the lightmatter workspace.")
matching_package_root = matching_notebook_path.parents[2]
matching_experimental_notebook = matching_notebook_path.with_name("experimental_data_analysis.ipynb")


def _matching_sources(path, markers):
    notebook = json.loads(path.read_text())
    sources = ["".join(cell["source"]) for cell in notebook["cells"]
               if cell["cell_type"] == "code"
               and not "".join(cell["source"]).startswith(("# Run the corrected single parameter case", "# Run the corrected CEP variation", "# Run the corrected exponent variation"))]
    selected = {}
    for stage, marker in markers.items():
        matches = [source for source in sources if marker in source]
        if len(matches) != 1:
            raise ValueError(f"Expected one cell containing {marker!r} in {path.name}; found {len(matches)}.")
        source = matches[0]
        guard = "# Notebook stage: executed by the final exponent runner.\nif not MATCHING_RUN_IN_FINAL_CELL:\n"
        if source.startswith(guard):
            source = textwrap.dedent(source[len(guard):])
        selected[stage] = source
    return selected


matching_simulation_sources = _matching_sources(matching_notebook_path, {
    "batch_settings": "# Shared corrected exponent-variation settings",
    "corrected_source_helpers": "# Corrected probe source provenance (read-only).",
    "pump_setup": "# Same geometry as the experimental scan;",
    "pump_run": "# Run this cell to simulate and save the pump.",
    "pump_plot": "# Show and save the pump propagation.",
    "probe_setup": "# Probe configuration and saved pump inputs",
    "helpers": "# Numerical helpers: spatial fields, cumulative integrals, and time shifts",
    "geometry": "# Geometry and interpolation cross-validation (no FDTD runs here)",
    "cache_and_duration": "# Build the shared normalized pump-response cache once.",
    "timing_diagnostic": "# Incident timing diagnostic and representative injection curves.",
    "geometry_plot": "# Simulation geometry diagnostic",
    "persistence": "# Compact result persistence and reusable material construction",
    "controls": "# Reference simulations: no carriers and the saturated spatial injection profile.",
    "delay_scan": "# Scan relative incident-peak delays.",
    "integrated_analysis": "# Load the compact scan outputs and compare integrated transmitted energy.",
    "lobe_setup": 'LOBE_SCAN_DIR = Path(base_dir) if "base_dir" in globals()',
    "lobe_helpers": "# Same lobe algorithm as experimental_data_analysis.ipynb;",
    "lobe_analysis": "# Stream completed saved runs; no simulation, experimental-data loading, or field rescaling.",
    "lobe_plots": "# Shaded reference and pumped lobes at representative saved delays.",
})
matching_experiment_sources = _matching_sources(matching_experimental_notebook, {
    "paths": "LIGHTMATTER_ROOT = next(",
    "validation": "def _paired_data(",
    "plot_helpers": "def _plot_analysis(",
    "pulse_helpers": "def complete_integral_analysis(",
    "lobe_helpers": "def fixed_cycle(",
    "settings": "# REQUIRED: choose the actual units used in your files.",
    "units": "unit_to_s = {",
    "pulse_analysis": "integrated = complete_integral_analysis(",
    "lobe_preview": "reference_preview = ref_waveforms[:, REFERENCE_INDEX]",
    "lobe_analysis": "lobe = fixed_cycle(",
    "results": "# Columns below retain the original experimental column order.",
    "comparison_settings": "# Simulation comparison settings",
    "comparison_helpers": "# Stored-output loading and reference-only alignment",
    "comparison_analysis": "# Analyze completed scans using the existing functions without rerunning FDTD.",
    "comparison_plots": "# Alignment preview: display scaling only, never used in energy normalization.",
    "comparison_export": "if SAVE_COMPARISON:",
})


def _matching_assignment(tree, name):
    matches = [node for node in tree.body if isinstance(node, ast.Assign)
               and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name)
               and node.targets[0].id == name]
    if len(matches) != 1:
        raise ValueError(f"Expected one top-level assignment to {name}.")
    return matches[0]


def _matching_compile(source, stage, overrides=None):
    tree = ast.parse(source)
    for name, value in (overrides or {}).items():
        _matching_assignment(tree, name).value = ast.parse(repr(value), mode="eval").body
    return compile(ast.fix_missing_locations(tree), f"CORRECTED_EXPONENT_VARIATION:{stage}", "exec")


# Read settings and compile every stage before loading fields or running FDTD.
matching_setup_tree = ast.parse(matching_simulation_sources["probe_setup"])
matching_settings_tree = ast.parse(matching_simulation_sources["batch_settings"])
matching_settings = {name: ast.literal_eval(_matching_assignment(matching_settings_tree, name).value)
                     for name in ("MATCHING_GAMMA_D0", "MATCHING_DEL_OMEGA_D", "MATCHING_PARENT_NAME", "MATCHING_PUMP_NAME", "MATCHING_PUMP_PHASE_RAD", "MATCHING_EXPONENT_CASES", "MATCHING_RUN_IN_FINAL_CELL")}
matching_gammas = matching_settings["MATCHING_GAMMA_D0"]
matching_delta = matching_settings["MATCHING_DEL_OMEGA_D"]
matching_output_dir = matching_package_root / "output/project2" / matching_settings["MATCHING_PARENT_NAME"]
matching_cases = matching_settings["MATCHING_EXPONENT_CASES"]
matching_figure_dir = matching_output_dir / "figures"
matching_comparison_dir = matching_output_dir / "experimental_simulation_comparison"
matching_simulation_code = {stage: _matching_compile(source, stage)
                            for stage, source in matching_simulation_sources.items()}
matching_experiment_code = {stage: _matching_compile(source, stage)
                            for stage, source in matching_experiment_sources.items()}
matching_probe_code = {
    gamma: _matching_compile(matching_simulation_sources["probe_setup"], "probe_setup",
                             {"gamma_D0": gamma, "del_omega_D": matching_delta})
    for gamma in matching_gammas
}
# Define only a read-only helper; this does not load the pump or run FDTD.
exec(matching_simulation_code["corrected_source_helpers"], globals())
matching_source_metadata, matching_source_provenance = load_corrected_probe_provenance(
    matching_notebook_path.parent
)
matching_lobe_code = _matching_compile(matching_simulation_sources["lobe_setup"], "lobe_setup",
                                       {"SAVE_LOBE_ANALYSIS": True})


def _matching_case_parent(case):
    return matching_output_dir / case["tag"]


def _matching_pump_directory(case=None):
    return matching_output_dir / matching_settings["MATCHING_PUMP_NAME"]


def _matching_case_namespace(case):
    return {
        "__name__": "__main__", **matching_settings,
        "MATCHING_CASE_PARENT": str(Path(matching_settings["MATCHING_PARENT_NAME"]) / case["tag"]),
        "MATCHING_PUMP_PARENT": matching_settings["MATCHING_PARENT_NAME"],
        "PUMP_FIELD_EXPONENT": case["exponent"],
        "MATCHING_PUMP_PHASE_RAD": 0.0,
        "MATCHING_PUMP_PHASE_LABEL": "0",
        "MATCHING_CASE_LABEL": case["label"],
        "load_corrected_probe_provenance": load_corrected_probe_provenance,
    }


def _matching_scan_directory(case, gamma=None):
    # Evaluate the producer's naming expression to preserve the conventional name.
    gamma = matching_gammas[0] if gamma is None else gamma
    namespace = {"package_root": matching_package_root, "gamma_D0": gamma,
                 "del_omega_D": matching_delta, **_matching_case_namespace(case)}
    namespace["SPATIAL_INTERPOLATION"] = ast.literal_eval(
        _matching_assignment(matching_setup_tree, "SPATIAL_INTERPOLATION").value)
    node = _matching_assignment(ast.parse(matching_simulation_sources["cache_and_duration"]), "base_dir")
    exec(compile(ast.Module(body=[node], type_ignores=[]), "scan_directory", "exec"), namespace)
    # Allow the output parent to be redirected for synthetic validation.
    return _matching_case_parent(case) / Path(namespace["base_dir"]).name


def _matching_validate_pump_inputs(pump_dir, phase_rad):
    """Validate the shared CEP=0 pump without loading the large E array."""
    import zipfile
    from scipy.constants import elementary_charge, h

    pump_dir = Path(pump_dir)
    for filename in ("info.json", "data.npz", "pump_metadata.json", "pump_material_probes.npz", "pump_propagation.png"):
        if not (pump_dir / filename).is_file():
            raise RuntimeError(f"Incomplete saved pump will not be overwritten: {pump_dir}; missing {filename}")
    info = json.loads((pump_dir / "info.json").read_text())
    metadata = json.loads((pump_dir / "pump_metadata.json").read_text())
    params, pulse = info["fdtd_params"], info["pulse_params"]
    reverse = matching_source_metadata["reverse_fdtd_params"]
    for key in ("Nz", "dz", "dt", "metal_i0", "metal_i1", "npml", "R_target", "m"):
        if not np.isclose(params[key], reverse[key], rtol=1e-12, atol=0):
            raise ValueError(f"Saved pump and corrected-source FDTD geometry/grid disagree for {key}.")
    for pump_key, reverse_key in (("src_i", "probe_t_i"), ("probe_t_i", "src_i"), ("probe_r_i", "probe_r_i")):
        if params[pump_key] != reverse[reverse_key]:
            raise ValueError(f"Saved pump and reverse calibration source/probe positions disagree for {pump_key}.")
    if (metadata.get("method") != "sio2_background"
            or not np.isclose(metadata["intensity_envelope_fwhm_fs"], 3.0)
            or not np.isclose(metadata["carrier_energy_eV"], 1.5)
            or not np.isclose(metadata.get("common_envelope_phase_rad", np.nan), phase_rad, rtol=0, atol=1e-12)):
        raise ValueError("Saved pump metadata must match CEP=0 and the 3 fs, 1.5 eV unpumped SiO2 configuration.")
    expected_pulse = {"E0": 1.0, "f0": 1.5*elementary_charge/h*1e-12,
                      "t0": 0.01, "tau": 3e-3/(2*np.sqrt(np.log(2))), "phase": phase_rad}
    for key, expected in expected_pulse.items():
        if not np.isclose(pulse.get(key, np.nan), expected, rtol=1e-12, atol=1e-15):
            raise ValueError(f"Saved pump pulse differs from the configured CEP source for {key}.")
    if pulse.get("type") != "gaussian":
        raise ValueError("Saved pump pulse must be Gaussian.")
    for key in ("background_del_eps", "background_lambda0_um", "background_gamma_per_ps"):
        actual, expected = np.asarray(metadata[key]), np.asarray(matching_source_metadata[key])
        if actual.shape != expected.shape or not np.allclose(actual, expected, rtol=1e-12, atol=0):
            raise ValueError(f"Saved pump and corrected-source background disagree for {key}.")
    headers = {}
    with zipfile.ZipFile(pump_dir / "pump_material_probes.npz") as archive:
        for name in ("E", "t_ps", "z_A", "source_E", "cell_indices"):
            with archive.open(name + ".npy") as stream:
                version = np.lib.format.read_magic(stream)
                if version == (1, 0):
                    shape, _, dtype = np.lib.format.read_array_header_1_0(stream)
                elif version == (2, 0):
                    shape, _, dtype = np.lib.format.read_array_header_2_0(stream)
                else:
                    raise ValueError(f"Unsupported saved pump array version {version} for {name}.")
                if dtype.hasobject:
                    raise ValueError("Saved pump arrays must not contain Python objects.")
                headers[name] = shape
    ntime = params["Nt"]
    nprobes = headers["z_A"][0] if len(headers["z_A"]) == 1 else 0
    if (ntime < 3 or nprobes < 1 or headers["t_ps"] != (ntime,)
            or headers["source_E"] != (ntime,) or headers["cell_indices"] != (nprobes,)
            or headers["E"] != (ntime, nprobes)):
        raise ValueError("Saved pump array headers disagree with its geometry or time axis.")
    with np.load(pump_dir / "pump_material_probes.npz", allow_pickle=False) as archive:
        times, source = archive["t_ps"], archive["source_E"]
        positions, indices = archive["z_A"], archive["cell_indices"]
    expected_source = pulse["E0"] * np.cos(2*np.pi*pulse["f0"]*(times-pulse["t0"])+phase_rad) * np.exp(
        -0.5*((times-pulse["t0"])/pulse["tau"])**2)
    if (not np.allclose(times, np.arange(ntime)*params["dt"], rtol=1e-12, atol=1e-15)
            or not np.allclose(source, expected_source, rtol=1e-12, atol=1e-14)
            or not np.array_equal(positions, indices*params["dz"])
            or np.any(np.diff(indices) <= 0)
            or np.any(indices < params["metal_i0"]) or np.any(indices >= params["metal_i1"])):
        raise ValueError("Saved pump source/axes do not match the requested CEP source and geometry.")
    return metadata


def _matching_expected_scan_settings():
    """Producer settings that must agree before saved fields can be reused."""
    names = {
        "omega_D0_rad_per_ps": "omega_D0", "interpolation": "SPATIAL_INTERPOLATION",
        "probe_source_peak_E0": "E0", "energy_tail_fraction": "ENERGY_TAIL_FRACTION",
        "scan_margin_fs": "SCAN_MARGIN_FS", "dense_main_overlap_samples": "N_DELAYS",
        "echo_delay_step_fs": "ECHO_DELAY_STEP_FS",
        "probe_occupancy_tail_fraction": "PROBE_OCCUPANCY_TAIL_FRACTION",
        "probe_timing_every_cells": "PROBE_TIMING_EVERY_CELLS",
        "minimum_post_record_margin_fs": "POST_RECORD_MARGIN_FS",
        "minimum_internal_round_trips": "MIN_INTERNAL_ROUND_TRIPS",
        "minimum_Drude_damping_times": "MIN_DRUDE_DAMPING_TIMES",
        "tail_check_fs": "TAIL_CHECK_FS", "tail_energy_tolerance": "TAIL_ENERGY_TOLERANCE",
    }
    expected = {key: ast.literal_eval(_matching_assignment(matching_setup_tree, name).value)
                for key, name in names.items()}
    expected["probe_source_baseline_subtracted_in_notebook"] = False
    return expected


def _matching_validate_saved_scan(run_dir, gamma, case):
    """Reject incomplete or mismatched data before refreshing any saved analysis."""
    run_dir = Path(run_dir)
    for filename in ("scan_configuration.json", "scan_manifest.json"):
        if not (run_dir / filename).is_file():
            raise RuntimeError(f"Incomplete saved scan will not be overwritten: {run_dir}; missing {filename}")
    config = json.loads((run_dir / "scan_configuration.json").read_text())
    if config.get("probe_source_provenance") != matching_source_provenance:
        raise RuntimeError(f"Saved scan has different or missing corrected-source provenance: {run_dir}")
    if (not np.isclose(config["gamma_D_per_ps"], gamma, rtol=0, atol=1e-12)
            or not np.isclose(config["del_omega_D_rad_per_ps"], matching_delta, rtol=0, atol=1e-12)
            or Path(config["pump_run_directory"]).resolve() != _matching_pump_directory(case).resolve()):
        raise ValueError(f"Scan parameters or shared pump directory do not match this exponent case: {run_dir}")
    if (not np.isclose(config.get("pump_common_envelope_phase_rad", np.nan), 0.0, rtol=0, atol=1e-12)
            or config.get("pump_common_envelope_phase_label") != "0"):
        raise ValueError(f"Saved scan CEP must be zero for every exponent case: {run_dir}")
    if config.get("pump_field_exponent") != case["exponent"]:
        raise ValueError(f"Saved scan exponent does not match this exponent case: {run_dir}")
    for key, expected in _matching_expected_scan_settings().items():
        if key not in config or config[key] != expected:
            raise ValueError(f"Saved scan configuration differs from current {key}: {run_dir}")
    saved = json.loads((run_dir / "scan_manifest.json").read_text())
    rows = saved.get("runs", [])
    indices = [row["index"] for row in rows]
    completed = saved.get("completed_indices", [])
    if (not saved.get("complete", False) or not rows or len(set(indices)) != len(indices)
            or len(set(completed)) != len(completed) or set(completed) != set(indices)):
        raise RuntimeError(f"Incomplete saved scan will not be overwritten: {run_dir}")
    for folder in ("unpumped_reference", "fully_injected_reference", *(row["folder"] for row in rows)):
        for filename in ("info.json", "data.npz"):
            if not (run_dir / folder / filename).is_file():
                raise FileNotFoundError(f"Completed scan is missing {run_dir / folder / filename}")
    for key in ("shared_axes", "shared_profile"):
        if not (run_dir / config[key]).is_file():
            raise FileNotFoundError(f"Completed scan is missing {run_dir / config[key]}")
    return config, saved


def _matching_preflight():
    """Plan all exponents and validate every existing artifact before output writes."""
    if matching_gammas != [250.0] or matching_delta != 130.0:
        raise ValueError("This notebook requires gamma_D0=250 and del_omega_D=130 for every exponent.")
    if (len(matching_cases) != 3
            or [case["exponent"] for case in matching_cases] != [4, 8, 12]
            or any(isinstance(case["exponent"], bool) or not isinstance(case["exponent"], int)
                   for case in matching_cases)
            or len({case["tag"] for case in matching_cases}) != 3
            or any(not case["tag"] or Path(case["tag"]).name != case["tag"]
                   or case["tag"] in (".", "..") for case in matching_cases)):
        raise ValueError("Configure three distinct exponent cases: E^4, E^8, and E^12.")
    if matching_settings["MATCHING_PUMP_PHASE_RAD"] != 0.0:
        raise ValueError("This exponent variation requires the default pump CEP=0.")
    metadata, provenance = load_corrected_probe_provenance(matching_notebook_path.parent)
    if provenance != matching_source_provenance or metadata != matching_source_metadata:
        raise RuntimeError("Corrected source changed after runner setup; restart and Run All.")
    for filename in ("time_axis.txt", "ref_field.txt", "mod_field.txt", "inj_time.txt"):
        if not (matching_notebook_path.parent / filename).is_file():
            raise FileNotFoundError(f"Missing experimental analysis input: {matching_notebook_path.parent / filename}")
    # A changed measurement must not trigger three scans with a stale correction.
    for filename, hash_key in (("ref_field.txt", "input_field_sha256"), ("time_axis.txt", "input_time_sha256")):
        source_path = matching_notebook_path.parent / filename
        digest = hashlib.sha256()
        with source_path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
        if digest.hexdigest() != metadata.get(hash_key):
            raise ValueError(f"Corrected source input provenance differs for {source_path}; regenerate the correction first.")
    plans = []
    gamma = matching_gammas[0]
    pump_dir = _matching_pump_directory()
    reuse_pump = pump_dir.exists()
    if reuse_pump:
        _matching_validate_pump_inputs(pump_dir, 0.0)
    for case in matching_cases:
        run_dir = _matching_scan_directory(case, gamma)
        reuse = run_dir.exists()
        if reuse:
            if not reuse_pump:
                raise RuntimeError(f"Saved scan is missing its shared CEP=0 pump: {pump_dir}")
            _matching_validate_saved_scan(run_dir, gamma, case)
        plans.append(dict(**case, gamma_D0_per_ps=gamma, scan_directory=str(run_dir),
                          pump_directory=str(pump_dir), case_parent=str(_matching_case_parent(case)),
                          reuse=reuse, reuse_pump=reuse_pump))
    return plans


def _matching_saved_analysis_namespace(run_dir, config):
    """Recreate only lightweight analysis inputs; never load the pump or solver."""
    from scipy.integrate import cumulative_trapezoid, trapezoid
    from tqdm.auto import tqdm

    def load_probe_run(folder):
        folder = Path(folder)
        info = json.loads((folder / "info.json").read_text())
        with np.load(folder / "data.npz", allow_pickle=False) as archive:
            data = {name: archive[name] for name in archive.files}
        if data["E_t"].shape != data["t_ps"].shape:
            raise ValueError("Saved trace and time-axis dimensions disagree.")
        return data, info

    with np.load(run_dir / config["shared_axes"], allow_pickle=False) as archive:
        waveform_time_rel_ps = archive["probe_source_t_rel_ps"]
        reference_field = archive["probe_source_E"]
    return dict(
        __name__="__main__", Path=Path, json=json, np=np, plt=plt, tqdm=tqdm,
        cumulative_trapezoid=cumulative_trapezoid, trapezoid=trapezoid, UNIT_T=1e-12,
        base_dir=run_dir, load_probe_run=load_probe_run,
        waveform_time_rel_ps=waveform_time_rel_ps, reference_field=reference_field,
        PUMP_FIELD_EXPONENT=config["pump_field_exponent"],
        MATCHING_PUMP_PHASE_RAD=0.0, MATCHING_PUMP_PHASE_LABEL="0",
        MATCHING_CASE_LABEL=f"E^{config['pump_field_exponent']}",
        probe_incident_peak_rel_ps=waveform_time_rel_ps[np.argmax(abs(reference_field))],
    )


def run_corrected_exponent_variation():
    """Run three exponents using one CEP=0 pump, then save individual and combined analyses."""
    plans = _matching_preflight()
    existing_figures = set(plt.get_fignums())

    def close_figures(*args, **kwargs):
        for number in set(plt.get_fignums()) - existing_figures:
            plt.close(number)

    experiment, namespace = {"__name__": "__main__"}, {}
    # Validate the experiment and all saved corrected sources before expensive work.
    with plt.ioff(), patch.object(plt, "show", close_figures):
        exec(matching_experiment_code["paths"], experiment)
        for variable, filename in (("time_ax", "time_axis.txt"), ("ref_waveforms", "ref_field.txt"),
                                   ("modified_waveforms", "mod_field.txt"), ("injection_times", "inj_time.txt")):
            experiment[variable] = np.genfromtxt(experiment["EXPERIMENTAL_DATA_DIR"] / filename, delimiter=",")
        for stage in ("validation", "plot_helpers", "pulse_helpers", "lobe_helpers", "settings", "units",
                      "pulse_analysis", "lobe_preview", "lobe_analysis", "results",
                      "comparison_settings", "comparison_helpers"):
            exec(matching_experiment_code[stage], experiment)
    for plan in plans:
        if plan["reuse"]:
            run_dir = Path(plan["scan_directory"])
            config, _ = _matching_validate_saved_scan(run_dir, plan["gamma_D0_per_ps"], plan)
            experiment["_simulation_source_peak"](run_dir, config)

    matching_output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = matching_output_dir / "grid_manifest.json"
    source_snapshot = dict(simulation=matching_simulation_sources, experiment=matching_experiment_sources)
    records = [dict(**plan, del_omega_D_rad_per_ps=matching_delta,
                    pump_common_envelope_phase_rad=0.0,
                    pump_common_envelope_phase_label="0",
                    pump_field_exponent=plan["exponent"],
                    pump_run_directory=plan["pump_directory"],
                    reused_completed_scan=plan["reuse"], status="pending",
                    pump_status="reused" if plan["reuse_pump"] else "pending") for plan in plans]
    manifest = {
        "complete": False, "gamma_D0_per_ps": matching_gammas,
        "del_omega_D_rad_per_ps": matching_delta,
        "pump_common_envelope_phase_rad": 0.0,
        "pump_common_envelope_phase_label": "0",
        "pump_field_exponents": [case["exponent"] for case in matching_cases],
        "dense_main_overlap_samples": ast.literal_eval(_matching_assignment(matching_setup_tree, "N_DELAYS").value),
        "parameter_conversion": "values used directly in existing solver units; no 2*pi factor",
        "pump_source": "one shared CEP=0 unpumped SiO2 propagation; reuse only validated complete pump outputs",
        "shared_pump_directory": str(_matching_pump_directory()),
        "shared_pump_status": "reused" if plans[0]["reuse_pump"] else "pending",
        "injection_normalization": "independent global sqrt(integral |E|^p dt) maximum for p=4,8,12, keeping del_omega_D fixed",
        "delay_definition": "probe incident abs(E) peak minus the shared CEP=0 sampled pump incident abs(E) peak at the sample front",
        "probe_source_provenance": matching_source_provenance,
        "comparison_status": "pending", "runs": records,
        "notebook": str(matching_notebook_path), "experimental_notebook": str(matching_experimental_notebook),
        "saved_source_sha256": hashlib.sha256(json.dumps(source_snapshot, sort_keys=True).encode()).hexdigest(),
        "comparison_directory": str(matching_comparison_dir), "figure_directory": str(matching_figure_dir),
    }

    def checkpoint():
        temporary = manifest_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(manifest, indent=2))
        temporary.replace(manifest_path)

    def copy_comparison_figures(source, destination):
        destination.mkdir(parents=True, exist_ok=True)
        for filename in ("reference_alignment.png", "experimental_simulated_energy_overplots.png",
                         "experimental_simulated_lobe_timing.png"):
            shutil.copy2(source / filename, destination / filename)

    (matching_output_dir / "executed_cell_sources.json").write_text(json.dumps(source_snapshot, indent=2))
    checkpoint()
    record = None
    try:
        with plt.ioff(), patch.object(plt, "show", close_figures):
            pump_dir = _matching_pump_directory()
            if plans[0]["reuse_pump"]:
                _matching_validate_pump_inputs(pump_dir, 0.0)
                print("Reusing the validated shared CEP=0 pump for all exponents.", flush=True)
            else:
                if pump_dir.exists():
                    raise FileExistsError(f"New pump directory appeared after preflight: {pump_dir}")
                namespace = _matching_case_namespace(plans[0])
                manifest["shared_pump_status"] = "running"
                for item in records:
                    item["pump_status"] = "running"
                checkpoint()
                print("Simulating the shared CEP=0 pump once for all exponents.", flush=True)
                exec(matching_simulation_code["pump_setup"], namespace)
                if Path(namespace["pump_output_dir"]).resolve() != pump_dir.resolve():
                    raise ValueError("The pump output directory differs from the planned shared pump directory.")
                for stage in ("pump_run", "pump_plot"):
                    exec(matching_simulation_code[stage], namespace)
                _matching_validate_pump_inputs(pump_dir, 0.0)
                manifest["shared_pump_status"] = "complete"
                for item in records:
                    item["pump_status"] = "complete"
                namespace.clear()
                close_figures()
                gc.collect()
                checkpoint()
            for plan, record in zip(plans, records):
                gamma = plan["gamma_D0_per_ps"]
                run_dir, pump_dir = Path(plan["scan_directory"]), Path(plan["pump_directory"])
                record.update(status="running", stage="probe")
                checkpoint()
                print(f"Pump exponent {plan['label']}, CEP=0: gamma_D0={gamma:g} ps^-1, del_omega_D={matching_delta:g} rad/ps", flush=True)
                if plan["reuse"]:
                    print("Reusing completed probe fields; refreshing waveform analyses.", flush=True)
                    config, saved = _matching_validate_saved_scan(run_dir, gamma, plan)
                    namespace = _matching_saved_analysis_namespace(run_dir, config)
                else:
                    if run_dir.exists():
                        raise FileExistsError(f"New scan directory appeared after preflight: {run_dir}")
                    namespace = _matching_case_namespace(plan)
                    exec(matching_probe_code[gamma], namespace)
                    for stage in ("helpers", "geometry", "cache_and_duration", "timing_diagnostic", "geometry_plot",
                                  "persistence", "controls", "delay_scan"):
                        record["stage"] = stage
                        checkpoint()
                        exec(matching_simulation_code[stage], namespace)
                    if Path(namespace["base_dir"]).resolve() != run_dir.resolve():
                        raise ValueError("The producer saved to a different scan directory than planned.")
                    config, saved = _matching_validate_saved_scan(run_dir, gamma, plan)
                record.update(number_of_delays=len(saved["runs"]), stage="analysis")
                checkpoint()
                for stage in ("integrated_analysis", "lobe_setup", "lobe_helpers", "lobe_analysis", "lobe_plots"):
                    code = matching_lobe_code if stage == "lobe_setup" else matching_simulation_code[stage]
                    exec(code, namespace)
                destination = matching_figure_dir / plan["tag"]
                destination.mkdir(parents=True, exist_ok=True)
                for relative in ("transmitted_energy_vs_peak_delay.png", "single_lobe_analysis/selected_lobe_previews.png",
                                 "single_lobe_analysis/lobe_energy_ratio_vs_delay.png", "single_lobe_analysis/selected_peak_time_vs_delay.png"):
                    shutil.copy2(run_dir / relative, destination / Path(relative).name)
                for source in run_dir.glob("*.png"):
                    shutil.copy2(source, destination / source.name)
                shutil.copy2(pump_dir / "pump_propagation.png", destination / "pump_propagation.png")
                record.update(figure_directory=str(destination), status="complete", stage="finished")
                checkpoint()
                namespace.clear()
                close_figures()
                gc.collect()

            experiment.update(
                SIMULATION_DIRECTORIES=[plan["scan_directory"] for plan in plans],
                SIMULATION_LABELS={str(Path(plan["scan_directory"]).resolve()):
                                   f"{plan['label']}; CEP=0; gamma_D0={plan['gamma_D0_per_ps']:g} /ps, delta omega_D={matching_delta:g} rad/ps"
                                   for plan in plans},
                SIMULATION_DATASET_NAMES={str(Path(plan["scan_directory"]).resolve()):
                                          plan["tag"] + "__" + Path(plan["scan_directory"]).name
                                          for plan in plans},
                ALLOW_PARTIAL_SCANS=False, SAVE_COMPARISON=True,
                COMPARISON_OUTPUT_DIR=matching_comparison_dir,
            )
            manifest["comparison_status"] = "running"
            checkpoint()
            exec(matching_experiment_code["comparison_analysis"], experiment)
            all_models = experiment["simulated_analyses"]
            models_by_path = {Path(model["folder"]).resolve(): model for model in all_models}
            for plan, record in zip(plans, records):
                per_case_comparison = Path(plan["case_parent"]) / "experimental_simulation_comparison"
                experiment.update(simulated_analyses=[models_by_path[Path(plan["scan_directory"]).resolve()]],
                                  COMPARISON_OUTPUT_DIR=per_case_comparison)
                for stage in ("comparison_plots", "comparison_export"):
                    exec(matching_experiment_code[stage], experiment)
                copy_comparison_figures(per_case_comparison,
                                        matching_figure_dir / plan["tag"] / per_case_comparison.name)
                record["comparison_directory"] = str(per_case_comparison)
                close_figures()
            experiment.update(simulated_analyses=all_models, COMPARISON_OUTPUT_DIR=matching_comparison_dir)
            for stage in ("comparison_plots", "comparison_export"):
                exec(matching_experiment_code[stage], experiment)
            copy_comparison_figures(matching_comparison_dir, matching_figure_dir / matching_comparison_dir.name)
            manifest.update(complete=True, comparison_status="complete")
    except BaseException as exc:
        manifest["error"] = f"{type(exc).__name__}: {exc}"
        if manifest["shared_pump_status"] == "running":
            manifest["shared_pump_status"] = "failed"
            for item in records:
                item["pump_status"] = "failed"
        if record is not None and record["status"] != "complete":
            record.update(status="failed", error=manifest["error"])
            if record["pump_status"] == "running":
                record["pump_status"] = "failed"
        if manifest["comparison_status"] == "running":
            manifest["comparison_status"] = "failed"
        raise
    finally:
        namespace.clear()
        experiment.clear()
        close_figures()
        gc.collect()
        checkpoint()
    print(f"Finished all three corrected exponent cases. Outputs: {matching_output_dir}\nFigures: {matching_figure_dir}")


run_corrected_exponent_variation()


Aligned waveform range: 0 to 80.056 fs
Sampling interval: 0.001 fs
Aligned injection range: -239.422 to 115.454 fs
Strongest eligible reference-peak times (fs):
[ 5.455 12.33  19.626 26.678 33.43  40.469 50.032 56.63  63.214 69.593]
Fixed positive target time for ALL pairs: 33.500000 fs
Valid lobe pairs: 48 / 48
Simulating the shared CEP=0 pump once for all exponents.
Number of material cells: 6410
Last probe id: 10703, last SiO2 cell:10705
Pump CEP: 0 (0 rad)
Carrier: 362.698386 THz, wavelength: 0.826561 um
Intensity FWHM: 3 fs; tau: 1.801684 fs
Grid: 15000 cells, 21780 steps, dt=0.006605 fs
Material probes: 713; field storage: 118.5 MiB
Output directory: /home/arch/MSC_studies/spring_26/semester_project/lightmatter/output/project2/corrected/exponent_variation/pump_3fs_1p5eV_sio2_12p82um
Simulation saved to: /home/arch/MSC_studies/spring_26/semester_project/lightmatter/output/project2/corrected/exponent_variation/pump_3fs_1p5eV_sio2_12p82um
Saved fields, material parameters, probes an

Computing sqrt(integral pump E^4):   0%|          | 0/201 [00:00<?, ?it/s]

Global sqrt(integral E^4) normalization: 0.0049364565 (field code units)^2 sqrt(ps)
Injection-response cache interpolation error <= 0.013% of global maximum
Largest final-5-fs pump injection fraction: 0.010%
Shared profile cache: 0.35 GiB
Probe bandwidth estimate: 88.85--237.22 THz; n_g=1.461673--1.490407
Incident front peaks at zero shift: probe=71.622 fs, saved pump=30.313 fs
Initial main-pulse overlap window: -64.962 to 46.310 fs
Recording margin: 63.734 fs (echo and damping allowances included)
Run duration: 257.849 fs, dt=0.006605 fs, Nt=39042
Reusable omega_D buffer: 1.86 GiB; not stored separately for each delay
last cell id : 6409
Simulation domain:       0.0000 to 30.0000 μm
Source position:         2.5000 μm
Reflection probe:        5.5900 μm
Transmission probe:      24.4100 μm
Sample region:            8.5900 to 21.4100 μm
Discretized sample width: 128200.0 Å
unpumped: last 10 fs energy fraction = 3.92e-05
fully injected: last 10 fs energy fraction = 2.92e-05
Final incident-

Pump-driven probe delay scan:   0%|          | 0/157 [00:00<?, ?run/s]

Saved 157 delay runs and shared injection data to /home/arch/MSC_studies/spring_26/semester_project/lightmatter/output/project2/corrected/exponent_variation/E4/probe_pump_sqrt_E4_3fs_1p5eV_gamma_D0_250.0000_ps-1_del_omega_D_130.0000_rad-ps-1_cubic


Integrating transmitted energy:   0%|          | 0/157 [00:00<?, ?it/s]

... (9 outputs left out)

Pump-driven probe delay scan:   0%|          | 0/157 [00:00<?, ?run/s]

Saved 157 delay runs and shared injection data to /home/arch/MSC_studies/spring_26/semester_project/lightmatter/output/project2/corrected/exponent_variation/E12/probe_pump_sqrt_E12_3fs_1p5eV_gamma_D0_250.0000_ps-1_del_omega_D_130.0000_rad-ps-1_cubic


Integrating transmitted energy:   0%|          | 0/157 [00:00<?, ?it/s]

Late-pump endpoint / unpumped energy: 1 (target 1)
Early-pump endpoint / unpumped energy: 0.961551
Fully injected reference / unpumped energy: 0.961551
Increase the scan margin if endpoint traces/energies have not approached their controls.
Strongest eligible reference-peak times (fs): [132.5603061  132.57351524 139.62059046 139.63379959 146.41008747
 153.44395355 153.45716269 162.9611376  162.98095131 162.99416045]
Analyzed 157 simulated traces; fixed positive target = 146.410087 fs.
Saved simulation-only lobe figures and numerical results to /home/arch/MSC_studies/spring_26/semester_project/lightmatter/output/project2/corrected/exponent_variation/E12/probe_pump_sqrt_E12_3fs_1p5eV_gamma_D0_250.0000_ps-1_del_omega_D_130.0000_rad-ps-1_cubic/single_lobe_analysis
probe_pump_sqrt_E4_3fs_1p5eV_gamma_D0_250.0000_ps-1_del_omega_D_130.0000_rad-ps-1_cubic: 157 runs, waveform shift=-112.9598 fs, reference correlation=0.9998
  Incident probe peak=31.308000 fs; valid lobes=157/157
  Comparison win